# 아기캥거루 통합 파이프라인 V6.0.5 — YOLO11 V6.0.5 FINAL 연동본

이 노트북은 `YOLO11_V6_0_5_FINAL.ipynb`이 게시하는
`pill_yolo_full_v6_0_preprocessed/`를 **단일 데이터 계약(source of truth)** 으로 사용한다.

## V6.0.5 연동 핵심 변경

1. 전처리 완료 상태는 `READY_FOR_TRAINING_PRESPLIT`만 허용한다.
2. 데이터 계약은 **12,196 images / 46,394 objects / 118 classes / 4,104 capture groups / 960×960**을 기준으로
   `preprocessing_report.json`, `dataset_manifest.csv`, 실제 이미지·YOLO 라벨을 교차 검증한다.
3. 전처리 산출물의 `images/train`, `images/val`, `labels/train`, `labels/val`을 그대로 사용한다.
4. **상류 Validation을 다시 Validation/Test로 쪼개지 않는다.** V6.0.5 FINAL의
   “Train/Val을 합치거나 재분할하지 않는다”는 계약을 그대로 지킨다.
5. 독립 labeled Test가 제공되지 않았으므로 이 노트북은 Test 점수를 만들지 않는다.
   후보 선택·모델 비교·시각화는 고정된 upstream Validation에서 수행하며,
   최종 리포트에 `independent_test_available=false`를 명시한다.
6. 세 모델(YOLO11s / YOLO11m / YOLO12m)은 완전히 같은 Train/Validation 파일과
   같은 class mapping을 사용한다.
7. Drive 원본 번들의 `dataset_manifest.csv`에 저장된 이미지/라벨 SHA-256을 로컬 staging 검증에 재사용해
   불필요한 원본 재해싱을 줄인다.

> 독립 Test 성능이 필요하면 별도의 group-safe labeled Test 데이터셋을 추가해야 한다.
> 현재 Validation을 다시 쪼개거나 Test로 재사용해 “최종 Test”라고 부르지 않는다.


# 분리 학습 전용 — YOLO11m tune_b

원본 V6.0.5의 공통 설정·데이터 검증·staging·학습 함수는 그대로 유지하고, 마지막 실행부만 **YOLO11m tune_b 1개**로 제한한 복사본이다.

- 다른 YOLO 학습 실험은 이 파일에서 실행하지 않는다.
- 체크포인트/manifest/timing 저장 경로와 학습 설정은 원본 함수를 그대로 사용한다.
- 학습 완료 후 이 노트북은 종료된다. 평가/최종 비교는 원본 통합 노트북에서 수행한다.


## V6.0.5 FINAL 연동 주의

> **[검토3]** 아래 V4/V5/추가검토 항목은 과거 수정 이력이며 현재 실행 정책이 아니다.
> 이번 검토에서 새로 고친 항목은 노트북 코드의 `[FIX A]`~`[FIX H]` 주석으로 표시했다.

아래 V4/V5 항목은 기존 버전의 변경 이력이다. 당시 Test 분리 관련 설명은 현재 실행 정책이 아니다.
현재 V6.0.5 연동본은 upstream Train/Validation을 그대로 보존하고 독립 Test를 생성하지 않는다.

## V4 핵심 수정사항

- 주 평가 지표: **mAP@[0.75:0.95]** (`0.75, 0.80, 0.85, 0.90, 0.95` 평균)
- 기존 COCO mAP50-95 / mAP50 / mAP75는 **진단용 보조 지표**로 유지
- Validation에서 confidence와 Top-K를 competition mAP로 공동 탐색
- 단일 Validation의 클래스 커버리지 한계를 표시하고 3개 group-safe split 진단 생성
- 12,196장 full-data 최종학습용 manifest 생성
- sample_submission.csv가 있으면 8개 컬럼 및 image_id 계약을 추가 검증
- EXIF orientation > 1이 발견되면 BBox 좌표계 육안 확인 전 release를 차단할 수 있는 gate 추가
- PNG는 이미 JPEG로 저장된 번들을 변환해도 손실이 복구되지 않으므로, **진짜 JPEG-vs-PNG A/B는 원본에서 전처리 번들을 다시 만들 때만 유효**하다는 안전장치를 명시한다.

## V5 정리 (이번 정리에서 고친 것)

- FRCNN 학습 함수 제거 이후 남아 있던 미정의 변수(`canonical_dataset` 등)로 노트북이 처음부터
  실행 불가였던 문제를 모두 고쳤다.
- `load_and_validate_yolo_bundle`이 서로 다른 시그니처로 3번 재정의·재실행되며 앞선 두 번의
  12,196장 SHA256 해싱 결과를 매번 폐기하던 구조적 중복을 제거하고 최종 버전 하나만 남겼다.
- 어디서도 호출되지 않는 죽은 함수(구버전 `compute_map_metrics`/`evaluate_records`,
  `select_confidence_on_validation`)와 미사용 import(`scipy.optimize.linear_sum_assignment`)를 삭제했다.
- **[이번 정리] `albumentations`가 설치만 되어 있어도 Ultralytics가 baseline/tuning 증강 설정과
  무관하게 자체 Blur/CLAHE 등 증강을 몰래 적용하던 문제**를 발견해 패키지 설치 자체를 제거했다
  (알약 각인 판독성을 해치지 않도록 증강을 직접 통제한다는 노트북의 설계 의도를 지키기 위함).
- **[이번 정리] `analyze_yolo_convergence()`가 계산하던 `needs_more_epochs` 진단이 계산만 되고
  어디에도 출력되지 않던 문제**를 고쳐, 학습 로그와 Validation leaderboard에 경고/열로 노출한다.
- **[이번 정리] 평가용 0..117 ID 매핑(`ORIGINAL_TO_EVAL`)이 학습용 YOLO ID 매핑
  (`ORIGINAL_TO_YOLO`)과 완전히 동일한 로직을 별도로 중복 구현하고 있던 부분**을 하나로 합쳐,
  두 매핑이 나중에 조용히 어긋날 위험을 없앴다.
- FRCNN 제거 후 남아 있던 주석/설명 문구 속 FRCNN 잔재 표현을 정리했다.

## 추가 검토 (이번 재검토에서 새로 고친 것)

- **[추가 검토] `## 2-3-1` 절 설명 문구에 FRCNN 비교 표현("Faster R-CNN과 같은")이 하나 더 남아 있던 것**을 찾아 제거했다 (다른 FRCNN 잔재는 이미 이전 V5 정리에서 정리됨).
- **[추가 검토] Test 클래스별 AP(`AP75_95`)를 원본 category ID로 되돌리는 로직이 `ORIGINAL_TO_YOLO`/`YOLO_TO_ORIGINAL`을 재사용하지 않고 `sorted(ORIGINAL_CLASS_INFO)`로 eval_id -> original_id 역매핑을 별도로 다시 계산하고 있던 부분**을 발견했다. 지금은 결과가 우연히 같지만 `ORIGINAL_TO_EVAL` 중복과 같은 종류의 위험이었으므로, 이미 있는 `YOLO_TO_ORIGINAL`을 그대로 재사용하도록 통일했다.
- **[추가 검토] Validation의 confidence x Top-K 정책 탐색(모델 9개 x 32가지 조합 = 최대 288회 호출)마다 실제로는 쓰지 않는 진단용 COCO mAP까지 매번 함께 계산하고 있던 것**을 확인했다. competition mAP만 계산하는 경량 함수(`compute_competition_map_only`)를 새로 분리해 그리드 탐색 단계의 불필요한 연산을 줄였다. Validation/Test 최종 리포트에 쓰이는 진단 지표는 기존 `compute_map_metrics()`가 그대로 전부 계산한다.
- **[추가 검토] `stage_shared_training_dataset()`가 반환하던 `YOLO_NAMES`가 data.yaml 생성 이후로는 어디에서도 쓰이지 않던 죽은 전역 변수였던 것**을 확인했다. 학습에 실제로 쓰인 "YOLO 인덱스 - original_category_id - 클래스 이름" 대조표를 manifest CSV로 저장해 재현성 문서에 실제로 활용한다.
- **[추가 검토] `build_group_train_val_test_split()`가 "그룹 2개 이상인데 그 그룹들이 (다른 singleton 클래스와 겹쳐서) 전부 강제로 Train에 배정된" 클래스를 만났을 때, 그 진단표(예전에는 `## 1-4` 앞의 별도 셀에서 9개 클래스 번호를 하드코딩해 확인)가 정작 크래시가 난 *뒤*에 있어서 원인을 못 보고 RuntimeError만 보게 되던 문제**를 고쳤다. `compute_group_class_structure()`/`report_forced_train_conflicts()`로 분리해 split 탐색 *전에* 118개 클래스 전체를 스캔한 충돌 진단표를 먼저 보여주도록 순서를 바꿨다(하드코딩된 9개 목록도 전체 스캔으로 일반화).
- **[추가 검토] 위 진단에서 발견되는 "구조적으로 평가 불가능한" 클래스(그룹이 몇 개든 늘려도, SPLIT_SEARCH_TRIALS를 아무리 키워도 Val/Test에 갈 그룹이 절대 생기지 않는 클래스) 때문에 `validate_final_split()`이 파이프라인 전체를 RuntimeError로 중단시키던 것**을 정책적으로 바꿨다. 이제 이런 클래스는 `eval_status="structurally_not_evaluable"`로 명시적으로 표시하고 경고만 남긴 뒤 학습·평가를 계속 진행한다. 반대로 "그룹은 남아 있었는데 이번 split이 우연히 못 채운" 경우는 split 탐색/점수 함수로 고칠 수 있는 문제이므로 여전히 하드 에러로 막는다 - 구조적 한계와 실제 버그를 구분해서, 데이터로만 해결 가능한 문제 때문에 전체 실행이 끝까지 막히는 일을 없애면서도 진짜 커버리지 버그는 계속 잡아낸다.


## 기본 구조

YOLO11 V6.0.5 FINAL 산출물 검증
→ **기존 Train/Validation 분할 그대로 보존**
→ 공통 로컬 학습 데이터 staging
→ YOLO11s / YOLO11m / YOLO12m 초기 학습
→ 모델별 파인튜닝
→ 고정 Validation에서 후보·confidence·Top-K 선택
→ 선택 모델 Validation 비교·시각화
→ 결과 저장·품질 체크

기본 실행 모드는 `standard`다. `smoke`는 연결과 코드 흐름 확인용이다.

**중요:** 이 번들에는 독립 labeled Test가 없으므로 이 노트북은 Test split을 새로 만들지 않는다.


# 0. 기본세팅

## 0-1. 재현 가능한 패키지 설치

Colab의 CUDA와 연결된 `torch`, `torchvision`은 기본 버전을 유지한다. 모델 동작과 평가 결과에 직접 영향을 주는 패키지만 고정하고, 실제 버전은 뒤에서 실험 기록에 같이 저장한다.

In [13]:
# 같은 노트북을 다시 실행해도 학습·평가 동작이 바뀌지 않도록 핵심 패키지 버전을 고정한다.
import subprocess
import sys

# YOLO11s·YOLO11m·YOLO12m 학습과 객체 탐지 평가를 지원하는 검증 버전이다.
# BUGFIX (V5 정리): albumentations는 노트북 어디에서도 import/직접 호출되지 않지만,
# 설치만 되어 있어도 Ultralytics는 model.train() 호출 시 자동으로 Blur/MedianBlur/
# ToGray/CLAHE 등 자체 확률(p≈0.01)의 albumentations 증강을 몰래 추가로 적용한다.
# 이는 hsv_h/degrees/... 값과 무관하게 켜지며, yolo_augmentation_arguments()의
# "baseline = 증강 없음(같은 출발 조건)", "safe_detection = 알약 각인 판독성을 해치지
# 않는 범위로만 제한된 증강"이라는 설계 의도를 조용히 깨뜨린다(특히 Blur/CLAHE는
# 알약 표면 각인 판독을 직접 방해할 수 있다). ultralytics는 model.train(augment=False)
# 로도 이 자동 albumentations 경로를 끌 수 없으므로(공식 이슈 #19807), 가장 안전한
# 해결책은 이 패키지 자체를 설치하지 않는 것이다 - Ultralytics의 Albumentations
# 래퍼는 import가 실패하면 스스로 비활성화된다.
PINNED_PACKAGES = [
    "ultralytics==8.4.116",
    "torchmetrics==1.9.0",
    "pycocotools==2.0.11",
    "tqdm==4.67.1",
]

# [검토2 수정] '설치하지 않는다'만으로는 부족하다. Colab 기본 이미지에 albumentations가 이미
# 깔려 있으면 Ultralytics가 그것을 감지해 자동 증강을 켜므로, 여기서 실제로 제거를 시도한다.
# (없으면 조용히 넘어간다. import 자체가 실패해야 Ultralytics 래퍼가 비활성화된다.)
_uninstall = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "albumentations"],
    capture_output=True, text=True,
)
try:
    import importlib
    if importlib.util.find_spec("albumentations") is not None:
        print("[WARNING] albumentations is still importable; "
              "Ultralytics may re-enable automatic augmentation. Restart the runtime if needed.")
    else:
        print("albumentations is not importable (automatic augmentation disabled).")
except Exception:
    pass

# 현재 Colab Python에 필요한 패키지를 설치한다.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES])
print("Package installation completed.")

albumentations is not importable (automatic augmentation disabled).
Package installation completed.


## 0-2. 라이브러리·Seed·환경 정보

In [2]:
# 표준 라이브러리는 파일, 해시, 설정, 시간과 안전한 복사에 사용한다.
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

# 데이터 처리, 증강, 통계와 시각화에 필요한 라이브러리를 불러온다.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
# [V5 정리] scipy.optimize.linear_sum_assignment(Hungarian 매칭)은 import만 되고
# 노트북 전체에서 한 번도 쓰이지 않았다 (TP/FP/FN 매칭은 confidence 내림차순 greedy 매칭인
# fixed_threshold_counts()로 구현돼 있음). 미사용 import라 삭제한다.
from tqdm.auto import tqdm

# PyTorch와 Torchvision의 객체 탐지 구성요소를 불러온다.
import torch
import torchvision
from torchvision.ops import box_iou

# 세 모델에 같은 COCO 방식 mAP를 적용하고 YOLO 모델을 실행한다.
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import ultralytics
from ultralytics import YOLO

# Colab Drive와 표 출력을 사용한다.
from google.colab import drive
from IPython.display import display

# 전처리 산출물과 체크포인트가 저장된 Google Drive를 연결한다.
drive.mount("/content/drive")

# Python, NumPy, PyTorch의 난수를 하나의 Seed로 고정한다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU가 있을 때 모든 CUDA 장치에도 같은 Seed를 적용한다.
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 재현성을 우선하고 지원되지 않는 결정론 연산은 경고만 표시한다.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 모든 모델이 같은 학습 장치를 사용한다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(torch.cuda.is_available())

# 실행 환경을 체크포인트와 최종 보고서에 남긴다.
RUNTIME_VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__,
    "torchmetrics": importlib_metadata.version("torchmetrics"),
    "pycocotools": importlib_metadata.version("pycocotools"),
    # BUGFIX (V5 정리): albumentations는 더 이상 설치하지 않는다(위 0-1 셀 참고).
    # 여기서 계속 importlib_metadata.version("albumentations")를 호출하면
    # 패키지가 없으므로 PackageNotFoundError로 노트북이 즉시 죽는다.
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda,
    "device": str(DEVICE),
}

print(f"Training device: {DEVICE}")
display(pd.Series(RUNTIME_VERSIONS, name="version").to_frame())

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
Training device: cuda


,version
python,3.12.13
torch,2.11.0+cu128
torchvision,0.26.0+cu128
ultralytics,8.4.116
torchmetrics,1.9.0
pycocotools,2.0.11
numpy,2.0.2
pandas,2.2.2
cuda,12.8
device,cuda


## 0-3. 공통 실험 설정과 경로

`YOLO11_V6_0_5_FINAL.ipynb`이 게시한
`pill_yolo_full_v6_0_preprocessed`를 그대로 연결한다.

- dataset state: `presplit_train_val_labeled_dataset`
- split: `train` / `val` 고정
- classes: 118
- preprocessing size: 960×960
- 재분할 금지


In [3]:
# ==============================================================================
# 1. 실행 및 YOLO11 V6.0.5 FINAL 계약
# ==============================================================================
RUN_PROFILE = "standard"  # "standard" 또는 "smoke"
PIPELINE_VERSION = "6.0.5"
PIPELINE_CODE_VERSION = "6.0.5-yolo11-final-presplit-exact"

# YOLO11 V6.0.5 FINAL이 선언한 고정 계약.
YOLO11_V605_CONTRACT = {
    "images": 12196,
    "objects": 46394,
    "classes": 118,
    "groups": 4104,
    "target_size": 960,
    "dataset_state": "presplit_train_val_labeled_dataset",
    "release_status": "READY_FOR_TRAINING_PRESPLIT",
}
EXPECTED_IMAGES = YOLO11_V605_CONTRACT["images"]
EXPECTED_OBJECTS = YOLO11_V605_CONTRACT["objects"]
EXPECTED_NUM_CLASSES = YOLO11_V605_CONTRACT["classes"]
EXPECTED_GROUPS = YOLO11_V605_CONTRACT["groups"]
EXPECTED_TARGET_SIZE = YOLO11_V605_CONTRACT["target_size"]

ACCEPTED_RELEASE_STATUS = {YOLO11_V605_CONTRACT["release_status"]}

# V6.0.5 FINAL은 기존 group-safe Train/Val 경계를 바꾸지 않는 것이 계약이다.
SPLIT_POLICY = "preserve_exact_upstream"
INDEPENDENT_TEST_AVAILABLE = False

# ==============================================================================
# 2. 모델 및 평가(Evaluation) 설정
# ==============================================================================
IMAGE_SIZE = EXPECTED_TARGET_SIZE
RAW_PREDICTION_CONFIDENCE = 0.001
COMMON_CONFIDENCE_THRESHOLD = 0.25
EVAL_IOU_THRESHOLD = 0.50

COMPETITION_METRIC = "mAP@[0.75:0.95]"
COMPETITION_IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
CONFIDENCE_CANDIDATES = [0.001, 0.01, 0.03, 0.05, 0.10, 0.20, 0.30, 0.50]
# [FIX F] 예전에는 여기서 TOP_K_CANDIDATES = None으로 두고, 같은 셀 아래쪽에서
# manifest의 max objects/image를 읽어 리스트로 조용히 덮어썼다. 위만 읽은 사람은
# 'Top-K 탐색이 꺼져 있다'고 오해한다. 실제 값이 정해지는 곳을 한 군데로 못 박는다.
TOP_K_CANDIDATES = None  # 아래 3-1 계약 검증 블록에서 max objects/image 기준으로 확정된다.

# Validation 규모가 크면 전 조합 grid 대신 coordinate search를 사용한다.
POLICY_SEARCH_MODE = "auto"
POLICY_GRID_IMAGE_LIMIT = 400

NMS_IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 300
COCO_EVAL_MAX_DETECTIONS = 100

# [FIX D] group-level bootstrap은 반복 1회마다 Validation 전체(수천 장)의 TP/FP/FN을
# 다시 센다. 500회 x 모델 3개면 Validation 2,433장 기준 수 시간짜리 작업이라 Colab
# 세션 안에서 끝나지 않는다. 데이터가 크면 반복수를 자동으로 낮추고, 실제로 몇 회를
# 돌았는지 최종 리포트에 기록해 CI 해석이 흐려지지 않게 한다.
BOOTSTRAP_ITERATIONS = 500
BOOTSTRAP_LARGE_SPLIT_IMAGE_THRESHOLD = 1000
BOOTSTRAP_ITERATIONS_LARGE_SPLIT = 150
VISUALIZATION_IMAGE_COUNT = 4

PROGRESS_BAR_FORMAT = (
    "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} "
    "[{elapsed}<{remaining}, {rate_fmt}]"
)

# ==============================================================================
# 3. 파일 및 디렉토리 경로
# ==============================================================================
SHARED_ROOT = Path("/content/drive/MyDrive/baby_kangaroo/week2")

PREPROCESS_ROOT = (
    SHARED_ROOT
    / "데이터전처리"
    / "yolo_전처리"
    / "pill_yolo_full_v6_0_preprocessed"
)

ARTIFACT_ROOT = SHARED_ROOT / "공통파이프라인"

# [검토 수정] class_split_feasibility.csv는 이 학습 노트북에서 읽지 않는다(upstream split을
# 그대로 보존하므로 split 가능성 진단이 필요 없음). 존재만 강제하던 죽은 요구사항이라 제거한다.
# dataset_version.json은 존재만 강제하고 버리는 대신 아래에서 실제로 열어 계약을 교차 검증한다.
REQUIRED_SOURCE_FILES = {
    "images": PREPROCESS_ROOT / "images",
    "labels": PREPROCESS_ROOT / "labels",
    "dataset_manifest.csv": PREPROCESS_ROOT / "dataset_manifest.csv",
    "class_mapping.csv": PREPROCESS_ROOT / "class_mapping.csv",
    "data.yaml": PREPROCESS_ROOT / "data.yaml",
    "dataset_version.json": PREPROCESS_ROOT / "dataset_version.json",
    "preprocessing_report.json": PREPROCESS_ROOT / "preprocessing_report.json",
}

# [FIX E] 세 모델(YOLO11s/11m/12m)이 완전히 같은 번들 하나를 공유하는데 키 이름만
# "YOLO11s"라서 '모델별 데이터'처럼 읽힌다. 실제 의미대로 공유 번들 키로 바꾼다.
SHARED_BUNDLE_KEY = "shared_v605_bundle"
YOLO_TRACK_DIRS = {SHARED_BUNDLE_KEY: PREPROCESS_ROOT}

for name, path in REQUIRED_SOURCE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required YOLO11 V6.0.5 preprocessing resource was not found: {name}\nPath: {path}"
        )

# ==============================================================================
# 3-1. V6.0.5 FINAL 계약을 report/manifest/mapping으로 교차 검증
# ==============================================================================
# [검토 수정] dataset_version.json은 지금까지 존재만 확인하고 내용을 한 번도 안 읽었다.
# 열어서 dataset_state가 계약(presplit_train_val_labeled_dataset)과 같은지 교차 검증한다.
_version_path = REQUIRED_SOURCE_FILES["dataset_version.json"]
_bundle_version = json.loads(_version_path.read_text(encoding="utf-8"))
_version_state = _bundle_version.get("dataset_state")
if _version_state is not None and _version_state != YOLO11_V605_CONTRACT["dataset_state"]:
    raise RuntimeError(
        f"dataset_version.json dataset_state mismatch: {_version_state!r}; "
        f"expected {YOLO11_V605_CONTRACT['dataset_state']!r}"
    )
# [검토2 수정] dataset_state만 보면 '어떤' presplit 번들이든 통과한다. 실제 upstream 버전이
# V6.0.5가 맞는지 버전 필드로 확인한다(키 이름이 번들마다 다를 수 있어 후보를 순회하고, 있으면 대조).
_version_value = next(
    (_bundle_version[_k] for _k in ("pipeline_version", "version", "dataset_version")
     if _k in _bundle_version),
    None,
)
# [FIX A] 실제 전처리 번들의 dataset_version.json이 쓰는 값은 순수 "6.0.5"가 아니라
# 'presplit-preprocessing-eda-v6.0.5' 같은 사람이 읽는 이름이다. 완전 일치 비교를 하면
# 정상 번들인데도 이 셀에서 RuntimeError가 나 노트북이 첫 화면에서 죽는다.
# 문자열 안에서 major.minor.patch를 뽑아 계약 버전과 대조한다(형식은 자유, 값은 엄격).
_version_match = re.search(r"(\d+\.\d+\.\d+)", str(_version_value or ""))
_version_normalized = _version_match.group(1) if _version_match else None
if _version_value is not None and _version_normalized != PIPELINE_VERSION:
    raise RuntimeError(
        f"dataset_version.json version mismatch: {_version_value!r} "
        f"(parsed={_version_normalized!r}); expected a V{PIPELINE_VERSION} bundle."
    )
print(f"dataset_version.json version check: {_version_value!r} -> {_version_normalized}")

_report_path = REQUIRED_SOURCE_FILES["preprocessing_report.json"]
_bundle_report = json.loads(_report_path.read_text(encoding="utf-8"))
_bundle_manifest = pd.read_csv(REQUIRED_SOURCE_FILES["dataset_manifest.csv"])
_bundle_mapping = pd.read_csv(REQUIRED_SOURCE_FILES["class_mapping.csv"])

if _bundle_report.get("release_status") != YOLO11_V605_CONTRACT["release_status"]:
    raise RuntimeError(
        f"Unexpected release_status: {_bundle_report.get('release_status')!r}; "
        f"expected {YOLO11_V605_CONTRACT['release_status']!r}"
    )
if _bundle_report.get("dataset_state") != YOLO11_V605_CONTRACT["dataset_state"]:
    raise RuntimeError(
        f"Unexpected dataset_state: {_bundle_report.get('dataset_state')!r}; "
        f"expected {YOLO11_V605_CONTRACT['dataset_state']!r}"
    )
if _bundle_report.get("image_preprocessing_applied") is not True:
    raise RuntimeError("YOLO11 V6.0.5 report does not confirm image preprocessing.")
if bool(_bundle_report.get("train_val_split_created", False)):
    raise RuntimeError(
        "The V6.0.5 bundle must preserve the upstream split; train_val_split_created must be False."
    )

if "split" not in _bundle_manifest.columns:
    raise RuntimeError("dataset_manifest.csv must contain the preserved 'split' column.")
_manifest_split_values = set(_bundle_manifest["split"].astype(str).str.lower())
if _manifest_split_values != {"train", "val"}:
    raise RuntimeError(
        f"V6.0.5 dataset_manifest split must be exactly train/val, found {_manifest_split_values}."
    )

_observed_contract = {
    "images": len(_bundle_manifest),
    "objects": int(_bundle_manifest["object_count"].sum()),
    "classes": len(_bundle_mapping),
    "groups": int(_bundle_manifest["group_id"].nunique()),
    "target_size": int(_bundle_report.get("preprocessing_target_size", -1)),
}
for key in ["images", "objects", "classes", "groups", "target_size"]:
    expected = YOLO11_V605_CONTRACT[key]
    observed = int(_observed_contract[key])
    report_value = {
        "images": _bundle_report.get("images"),
        "objects": _bundle_report.get("objects"),
        "classes": _bundle_report.get("classes"),
        "groups": _bundle_report.get("groups"),
        "target_size": _bundle_report.get("preprocessing_target_size"),
    }[key]
    if report_value is not None and int(report_value) != expected:
        raise RuntimeError(
            f"V6.0.5 report contract mismatch for {key}: {report_value} != {expected}"
        )
    if observed != expected:
        raise RuntimeError(
            f"V6.0.5 manifest/mapping contract mismatch for {key}: {observed} != {expected}"
        )

# 같은 group_id가 train과 val에 동시에 있으면 upstream 계약 자체가 깨진 것이다.
_group_split_nunique = _bundle_manifest.groupby("group_id")["split"].nunique()
if int((_group_split_nunique > 1).sum()) != 0:
    raise RuntimeError("Capture-group leakage already exists in the upstream V6.0.5 manifest.")

_max_objects_per_image = int(_bundle_manifest["object_count"].max())
TOP_K_CANDIDATES = sorted({
    max(1, _max_objects_per_image),
    max(1, _max_objects_per_image * 2),
    max(1, _max_objects_per_image * 3),
}) + [None]

print("=" * 72)
print("YOLO11 V6.0.5 FINAL DATASET CONTRACT")
print("=" * 72)
print(f"release_status     : {_bundle_report.get('release_status')}")
print(f"dataset_state      : {_bundle_report.get('dataset_state')}")
print(f"images             : {EXPECTED_IMAGES:,}")
print(f"objects            : {EXPECTED_OBJECTS:,}")
print(f"classes            : {EXPECTED_NUM_CLASSES:,}")
print(f"capture groups     : {EXPECTED_GROUPS:,}")
print(f"target image size  : {EXPECTED_TARGET_SIZE}")
print(f"max objects/image  : {_max_objects_per_image}")
print(f"Top-K candidates   : {TOP_K_CANDIDATES}")
print(f"split policy       : {SPLIT_POLICY}")
print("independent test   : unavailable (no downstream re-split)")

CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"
MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
REPORT_DIR = ARTIFACT_ROOT / "reports"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
BACKUP_DIR = ARTIFACT_ROOT / "incompatible_backups"
TIMING_DIR = ARTIFACT_ROOT / "timings"

# Colab local staging
LOCAL_WORK_ROOT = Path("/content/baby_kangaroo_week2_common_pipeline_v605")
LOCAL_IMAGE_ROOT = LOCAL_WORK_ROOT / "shared_images"
LOCAL_YOLO_DATASET = LOCAL_WORK_ROOT / "yolo_common"
LOCAL_RUNS_ROOT = LOCAL_WORK_ROOT / "runs"

LOCAL_SPLIT_TRAIN_IMAGES = LOCAL_YOLO_DATASET / "images" / "train"
LOCAL_SPLIT_VAL_IMAGES = LOCAL_YOLO_DATASET / "images" / "val"
LOCAL_SPLIT_TRAIN_LABELS = LOCAL_YOLO_DATASET / "labels" / "train"
LOCAL_SPLIT_VAL_LABELS = LOCAL_YOLO_DATASET / "labels" / "val"

directories_to_create = [
    CHECKPOINT_DIR, MANIFEST_DIR, REPORT_DIR, TIMING_DIR, FIGURE_DIR, BACKUP_DIR,
    LOCAL_WORK_ROOT, LOCAL_RUNS_ROOT, LOCAL_YOLO_DATASET,
    LOCAL_SPLIT_TRAIN_IMAGES, LOCAL_SPLIT_VAL_IMAGES,
    LOCAL_SPLIT_TRAIN_LABELS, LOCAL_SPLIT_VAL_LABELS,
]
for directory in directories_to_create:
    directory.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN_EXPERIMENTS = set()

if RUN_PROFILE not in {"standard", "smoke"}:
    raise ValueError("RUN_PROFILE must be 'standard' or 'smoke'.")
if SPLIT_POLICY != "preserve_exact_upstream":
    raise ValueError("YOLO11 V6.0.5 integration requires SPLIT_POLICY='preserve_exact_upstream'.")
if POLICY_SEARCH_MODE not in {"grid", "coordinate", "auto"}:
    raise ValueError("POLICY_SEARCH_MODE must be 'grid', 'coordinate', or 'auto'.")
if IMAGE_SIZE != EXPECTED_TARGET_SIZE:
    raise ValueError("IMAGE_SIZE must match the V6.0.5 preprocessing target size.")
if RUN_PROFILE == "standard" and not torch.cuda.is_available():
    raise RuntimeError("The standard training profile requires a Colab GPU runtime.")

print("\nYOLO11 V6.0.5 preprocessing source:")
for name, path in REQUIRED_SOURCE_FILES.items():
    print(f"{name:30s}: {path}")


dataset_version.json version check: '6.0.5' -> 6.0.5
YOLO11 V6.0.5 FINAL DATASET CONTRACT
release_status     : READY_FOR_TRAINING_PRESPLIT
dataset_state      : presplit_train_val_labeled_dataset
images             : 12,196
objects            : 46,394
classes            : 118
capture groups     : 4,104
target image size  : 960
max objects/image  : 6
Top-K candidates   : [6, 12, 18, None]
split policy       : preserve_exact_upstream
independent test   : unavailable (no downstream re-split)

YOLO11 V6.0.5 preprocessing source:
images                        : /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/images
labels                        : /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/labels
dataset_manifest.csv          : /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/dataset_manifest.csv
class_mapping.csv             : /content/drive/MyDrive/baby_kan

## 0-4. 파일 지문·manifest·안전한 백업 공용 함수

In [4]:
def stable_json_hash(value):
    """딕셔너리 순서와 관계없이 같은 JSON 내용에 같은 SHA256 지문을 만든다."""
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    """큰 이미지와 체크포인트를 메모리에 한꺼번에 올리지 않고 SHA256을 계산한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    """런타임 중단 때 불완전한 JSON이 남지 않도록 임시 파일을 완성한 뒤 교체한다."""
    path = Path(path)

    # 상위 폴더 보장
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    temporary_path.replace(path)


def backup_file(path, reason):
    """비호환 파일은 삭제하지 않고 해시가 포함된 이름으로 보존한다."""
    path = Path(path)
    if not path.is_file():
        return None
    BACKUP_DIR.mkdir(
        parents=True,
        exist_ok=True
    )
    short_hash = sha256_file(path)[:10]
    safe_reason = re.sub(r"[^A-Za-z0-9_-]+", "_", reason).strip("_")
    destination = BACKUP_DIR / f"{path.stem}_{safe_reason}_{short_hash}{path.suffix}"
    if not destination.exists():
        shutil.copy2(path, destination)
    print(f"Backed up incompatible file: {destination.name}")
    return destination


def build_checkpoint_manifest(model_name, base_weights, dataset_fingerprint, split_fingerprint, config):
    """체크포인트를 만든 데이터·분할·환경·학습 설정을 하나의 신분증으로 만든다."""
    manifest = {
        "schema_version": 5,
        "pipeline_version": PIPELINE_VERSION,
        "pipeline_code_version": PIPELINE_CODE_VERSION,
        "model_name": model_name,
        "base_weights": base_weights,
        "dataset_fingerprint": dataset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "runtime_versions": RUNTIME_VERSIONS,
        "split_config": {
            "policy": SPLIT_POLICY,
            "source": "YOLO11_V6_0_5_FINAL dataset_manifest.csv",
            "independent_test_available": INDEPENDENT_TEST_AVAILABLE,
            "seed_affects_split": False,
            "expected_images": EXPECTED_IMAGES,
            "expected_objects": EXPECTED_OBJECTS,
            "expected_num_classes": EXPECTED_NUM_CLASSES,
            "expected_groups": EXPECTED_GROUPS,
        },

        "training_config": config,
    }
    manifest["run_signature"] = stable_json_hash(manifest)[:16]
    return manifest


def save_checkpoint_manifest(checkpoint_path, manifest_path, expected_manifest):
    """학습 완료 상태와 체크포인트 해시를 manifest에 기록한다."""
    stored_manifest = dict(expected_manifest)
    stored_manifest["training_completed"] = True
    stored_manifest["checkpoint_sha256"] = sha256_file(checkpoint_path)
    write_json_atomic(manifest_path, stored_manifest)


def checkpoint_is_compatible(checkpoint_path, manifest_path, expected_manifest):
    """완료된 체크포인트가 현재 실험 계약과 정확히 같을 때만 재사용한다."""
    checkpoint_path = Path(checkpoint_path)
    manifest_path = Path(manifest_path)
    if not checkpoint_path.is_file() or not manifest_path.is_file():
        return False

    try:
        saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False

    saved_checkpoint_hash = saved_manifest.pop("checkpoint_sha256", None)
    training_completed = bool(saved_manifest.pop("training_completed", False))
    if not training_completed:
        return False

    if saved_checkpoint_hash is None:
        return False

    if (
        saved_checkpoint_hash
        != sha256_file(checkpoint_path)
    ):
        return False

    # schema 자체가 다르면 재사용하지 않는다.
    if (
        saved_manifest.get("schema_version")
        != expected_manifest.get("schema_version")
    ):
        return False

    return saved_manifest == expected_manifest


def reset_owned_directory(directory, allowed_root):
    """이 파이프라인 전용 로컬 폴더만 초기화해 잘못된 경로 삭제를 막는다."""
    directory = Path(directory).resolve()
    allowed_root = Path(allowed_root).resolve()
    if directory == allowed_root or allowed_root not in directory.parents:
        raise RuntimeError(f"Deletion is not allowed outside the owned root: {directory}")
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def json_ready(value):
    """NumPy·Pandas·Path 값을 JSON으로 저장 가능한 기본 자료형으로 바꾼다."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    # [V6 수정] np.bool_을 float으로 바꾸면 True가 1.0으로 저장돼 리포트 JSON에서
    # 불리언 필드가 숫자로 둔갑한다.
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


# 1. 데이터 - 결과물 받는 자리

## 1-0. YOLO11 V6.0.5 전처리 계약과 고정 Train/Validation 분할

`dataset_manifest.csv`의 `split`과 `group_id`를 그대로 사용한다.

- Train/Validation을 다시 나누지 않는다.
- 동일 group_id가 두 split에 동시에 존재하면 즉시 실패한다.
- 파일 누락·중복, 객체 수, 클래스 수, 그룹 수를 전체 검증한다.
- 독립 Test는 만들지 않는다.


In [5]:
def read_preprocessing_report(bundle_dir):
    """YOLO11 V6.0.5 FINAL preprocessing_report.json을 엄격하게 검증한다."""
    bundle_dir = Path(bundle_dir)
    report_path = bundle_dir / "preprocessing_report.json"
    if not report_path.is_file():
        raise FileNotFoundError(f"Preprocessing report was not found: {report_path}")

    report = json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("release_status") not in ACCEPTED_RELEASE_STATUS:
        raise RuntimeError(
            f"Preprocessing release is not ready: status={report.get('release_status')!r}"
        )
    if report.get("dataset_state") != YOLO11_V605_CONTRACT["dataset_state"]:
        raise RuntimeError(
            f"Unexpected dataset_state: {report.get('dataset_state')!r}"
        )
    if report.get("image_preprocessing_applied") is not True:
        raise RuntimeError("Image preprocessing was not confirmed.")
    if int(report.get("preprocessing_target_size", -1)) != EXPECTED_TARGET_SIZE:
        raise RuntimeError("Unexpected preprocessing target size.")
    for key, report_key in [
        ("images", "images"), ("objects", "objects"),
        ("classes", "classes"), ("groups", "groups"),
    ]:
        if int(report.get(report_key, -1)) != int(YOLO11_V605_CONTRACT[key]):
            raise RuntimeError(
                f"V6.0.5 report mismatch: {report_key}={report.get(report_key)!r}, "
                f"expected={YOLO11_V605_CONTRACT[key]}"
            )
    # [검토 수정] group_contract_sha256 / label_contract_sha256는 split_manifest(1-4 셀)와
    # 최종 리포트(4-2 셀)에서 canonical_dataset["report"][...]로 반드시 참조된다. 여기서 존재를
    # 함께 검증하지 않으면, 이 키가 없는 번들도 데이터 검증은 통과한 뒤 한참 뒤 단계에서 raw
    # KeyError로 죽어 원인 파악이 어렵다. report를 읽는 이 지점에서 계약 위반으로 조기 실패시킨다.
    for contract_key in ("group_contract_sha256", "label_contract_sha256"):
        value = report.get(contract_key)
        if not isinstance(value, str) or not value.strip():
            raise RuntimeError(
                f"V6.0.5 report is missing required fingerprint: {contract_key!r}. "
                "The preprocessing bundle must publish this contract hash."
            )
    return report_path, report


def read_dataset_manifest(bundle_dir):
    """V6.0.5 dataset_manifest.csv와 고정 Train/Val group 계약을 검증한다."""
    manifest_path = Path(bundle_dir) / "dataset_manifest.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Dataset manifest was not found: {manifest_path}")

    manifest_df = pd.read_csv(manifest_path)
    required_columns = {
        "file_name", "split", "group_id", "object_count",
        "image_sha256", "label_sha256",
    }
    missing_columns = required_columns - set(manifest_df.columns)
    if missing_columns:
        raise ValueError(f"Dataset manifest columns are missing: {sorted(missing_columns)}")

    # [검토2 수정] 결측(NaN) 검사는 반드시 문자열 변환 '전에' 해야 한다. astype(str)를 먼저 하면
    # NaN이 문자열 'nan'으로 바뀌어 isna()가 영원히 False가 되고 결측 검사가 무력화된다.
    # file_name/group_id/object_count 모두 변환 전에 결측을 잡고, 문자열 컬럼은 변환 후 공백도 잡는다.
    for required_notnull in ["file_name", "group_id", "object_count"]:
        if manifest_df[required_notnull].isna().any():
            raise ValueError(f"Missing values were found in dataset_manifest column: {required_notnull}")

    manifest_df = manifest_df.copy()
    manifest_df["file_name"] = manifest_df["file_name"].astype(str).str.strip()
    manifest_df["split"] = manifest_df["split"].astype(str).str.lower().str.strip()
    manifest_df["group_id"] = manifest_df["group_id"].astype(str).str.strip()

    for non_empty in ["file_name", "group_id"]:
        if manifest_df[non_empty].eq("").any():
            raise ValueError(f"Empty {non_empty} values were found in dataset_manifest.")

    if manifest_df["file_name"].duplicated().any():
        duplicates = manifest_df.loc[
            manifest_df["file_name"].duplicated(keep=False), "file_name"
        ].tolist()[:10]
        raise ValueError(f"Duplicate file names were found: {duplicates}")
    if set(manifest_df["split"]) != {"train", "val"}:
        raise RuntimeError(
            f"Expected exactly train/val in manifest split, found {sorted(set(manifest_df['split']))}"
        )

    mixed_groups = (
        manifest_df.groupby("group_id")["split"].nunique().gt(1)
    )
    if mixed_groups.any():
        raise RuntimeError(
            f"{int(mixed_groups.sum())} capture group(s) cross Train/Validation upstream boundary."
        )

    observed = {
        "images": len(manifest_df),
        "objects": int(manifest_df["object_count"].sum()),
        "groups": int(manifest_df["group_id"].nunique()),
    }
    expected = {
        "images": EXPECTED_IMAGES,
        "objects": EXPECTED_OBJECTS,
        "groups": EXPECTED_GROUPS,
    }
    if observed != expected:
        raise RuntimeError(f"Manifest contract mismatch: observed={observed}, expected={expected}")
    return manifest_df.sort_values("file_name").reset_index(drop=True)


def build_upstream_split_by_group(manifest_df):
    """group_id -> train/val 고정 매핑."""
    table = manifest_df[["group_id", "split"]].copy()
    table["group_id"] = table["group_id"].astype(str)
    table["split"] = table["split"].astype(str).str.lower()
    mixed = table.groupby("group_id")["split"].nunique().gt(1)
    if mixed.any():
        raise RuntimeError("Upstream group leakage detected.")
    return (
        table.drop_duplicates("group_id")
        .set_index("group_id")["split"]
        .to_dict()
    )


def compute_group_class_structure(coco, file_to_group):
    """고정 split 진단용 그룹/클래스 구조를 계산한다."""
    image_by_id = {int(image["id"]): image for image in coco["images"]}
    group_files = defaultdict(list)
    group_class_counts = defaultdict(Counter)
    class_groups = defaultdict(set)

    for image in coco["images"]:
        file_name = image["file_name"]
        if file_name not in file_to_group:
            raise KeyError(f"Group ID is missing for {file_name}")
        group_files[file_to_group[file_name]].append(file_name)

    for annotation in coco["annotations"]:
        image = image_by_id[int(annotation["image_id"])]
        group_id = file_to_group[image["file_name"]]
        category_id = int(annotation["category_id"])
        group_class_counts[group_id][category_id] += 1
        class_groups[category_id].add(group_id)

    if len(group_files) != EXPECTED_GROUPS:
        raise RuntimeError(f"Expected {EXPECTED_GROUPS} groups, found {len(group_files)}.")
    if len(coco["images"]) != EXPECTED_IMAGES:
        raise RuntimeError(f"Expected {EXPECTED_IMAGES} images, found {len(coco['images'])}.")
    if len(coco["annotations"]) != EXPECTED_OBJECTS:
        raise RuntimeError(f"Expected {EXPECTED_OBJECTS} objects, found {len(coco['annotations'])}.")
    if len(class_groups) != EXPECTED_NUM_CLASSES:
        raise RuntimeError(f"Expected {EXPECTED_NUM_CLASSES} classes, found {len(class_groups)}.")

    singleton_classes = {
        category_id for category_id, groups in class_groups.items() if len(groups) == 1
    }
    return group_files, group_class_counts, class_groups, singleton_classes


def validate_exact_upstream_split(split_files, split_groups, file_to_group, coco):
    """V6.0.5 upstream Train/Val이 누수 없이 전 파일을 정확히 한 번 포함하는지 검사한다."""
    if set(split_files) != {"train", "val"}:
        raise RuntimeError(f"Only train/val are allowed, found {sorted(split_files)}.")

    train_files = set(split_files["train"])
    val_files = set(split_files["val"])
    if train_files & val_files:
        raise RuntimeError("File leakage detected between Train and Validation.")

    expected_files = {image["file_name"] for image in coco["images"]}
    assigned_files = train_files | val_files
    if assigned_files != expected_files:
        raise RuntimeError(
            f"Upstream split completeness mismatch: missing={len(expected_files-assigned_files)}, "
            f"extra={len(assigned_files-expected_files)}"
        )

    if set(split_groups["train"]) & set(split_groups["val"]):
        raise RuntimeError("Capture-group leakage detected between Train and Validation.")

    # file_to_group로 한 번 더 독립 검증
    train_groups_from_files = {file_to_group[name] for name in train_files}
    val_groups_from_files = {file_to_group[name] for name in val_files}
    if train_groups_from_files != set(split_groups["train"]):
        raise RuntimeError("Train group set does not match file_to_group mapping.")
    if val_groups_from_files != set(split_groups["val"]):
        raise RuntimeError("Validation group set does not match file_to_group mapping.")


def build_exact_upstream_split(coco, file_to_group, manifest_df, verbose=True):
    """dataset_manifest.csv의 기존 Train/Val을 한 장도 이동시키지 않고 사용한다."""
    split_files = {
        split_name: sorted(
            manifest_df.loc[manifest_df["split"].eq(split_name), "file_name"].astype(str)
        )
        for split_name in ["train", "val"]
    }
    split_groups = {
        split_name: set(
            manifest_df.loc[manifest_df["split"].eq(split_name), "group_id"].astype(str)
        )
        for split_name in ["train", "val"]
    }

    _group_files, _group_class_counts, class_groups, singleton_classes = (
        compute_group_class_structure(coco, file_to_group)
    )
    validate_exact_upstream_split(split_files, split_groups, file_to_group, coco)

    image_by_id = {int(image["id"]): image for image in coco["images"]}
    image_to_split = {
        file_name: split_name
        for split_name, names in split_files.items()
        for file_name in names
    }
    class_split_counts = defaultdict(Counter)
    for annotation in coco["annotations"]:
        image = image_by_id[int(annotation["image_id"])]
        class_split_counts[int(annotation["category_id"])][
            image_to_split[image["file_name"]]
        ] += 1

    coverage_rows = []
    for category_id in sorted(class_groups):
        counts = class_split_counts[category_id]
        coverage_rows.append({
            "category_id": category_id,
            "group_count": len(class_groups[category_id]),
            "train_objects": int(counts["train"]),
            "val_objects": int(counts["val"]),
            "is_single_group_class": category_id in singleton_classes,
            "validation_status": "observed" if counts["val"] > 0 else "not_observed",
        })
    coverage_table = pd.DataFrame(coverage_rows)
    if len(coverage_table) != EXPECTED_NUM_CLASSES:
        raise RuntimeError(
            f"Expected {EXPECTED_NUM_CLASSES} classes in coverage table, found {len(coverage_table)}."
        )

    if verbose:
        print("=== YOLO11 V6.0.5 upstream split preserved exactly ===")
        print(
            f"Train: {len(split_groups['train']):,} groups / {len(split_files['train']):,} images"
        )
        print(
            f"Val  : {len(split_groups['val']):,} groups / {len(split_files['val']):,} images"
        )
        print("Independent Test: not provided; no downstream split was fabricated.")
        display(coverage_table)

    # split_score는 이전 버전과의 변수 호환을 위해 0.0으로 둔다.
    return split_files, class_groups, 0.0, split_groups, coverage_table


## 1-1. YOLO11 V6.0.5 FINAL 데이터 전체 검증

`preprocessing_report.json`, `dataset_manifest.csv`, `class_mapping.csv`,
실제 `images/{train,val}`과 `labels/{train,val}`을 교차 검증한다.

검증 기준:
- 12,196 images
- 46,394 objects
- 118 classes
- 4,104 capture groups
- 960×960
- `READY_FOR_TRAINING_PRESPLIT`
- `presplit_train_val_labeled_dataset`
- upstream Train/Validation 경계 완전 보존


In [6]:
def read_yolo_mapping(mapping_path):
    """YOLO 0..117 ID와 실제 original_category_id의 1:1 매핑을 검증한다."""
    mapping_df = pd.read_csv(mapping_path)
    required_columns = {"yolo_class_id", "original_category_id", "normalized_class_name"}
    missing_columns = required_columns - set(mapping_df.columns)
    if missing_columns:
        raise ValueError(f"YOLO mapping columns are missing: {sorted(missing_columns)}")

    mapping_df = mapping_df.copy()
    # [검토2 수정] astype(int)는 3.7을 조용히 3으로 절삭한다. YOLO 라벨 검증과 같은 수준으로,
    # numeric 변환 후 '정수인지'를 먼저 확인하고 나서 int로 캐스팅한다(비정수는 명확히 실패).
    for id_column in ["yolo_class_id", "original_category_id"]:
        numeric_ids = pd.to_numeric(mapping_df[id_column], errors="raise")
        if not np.isfinite(numeric_ids).all():
            raise ValueError(f"Non-finite value in class_mapping column: {id_column}")
        if not (numeric_ids == numeric_ids.round()).all():
            raise ValueError(f"Non-integer value in class_mapping column: {id_column}")
        mapping_df[id_column] = numeric_ids.astype(int)
    mapping_df = mapping_df.sort_values("yolo_class_id").reset_index(drop=True)

    if mapping_df["yolo_class_id"].tolist() != list(range(EXPECTED_NUM_CLASSES)):
        raise ValueError(
            f"YOLO IDs must be continuous from 0 to {EXPECTED_NUM_CLASSES - 1}."
        )
    if mapping_df["original_category_id"].duplicated().any():
        raise ValueError("Duplicate original category IDs were found in class_mapping.csv.")
    if mapping_df["normalized_class_name"].fillna("").astype(str).str.strip().eq("").any():
        raise ValueError("Empty normalized class names were found in class_mapping.csv.")

    idx_to_original = dict(zip(
        mapping_df["yolo_class_id"].astype(int),
        mapping_df["original_category_id"].astype(int),
    ))
    return mapping_df, idx_to_original


def validate_yolo_label_file(label_path):
    """YOLO TXT의 정수 class ID와 normalized bbox를 엄격 검증한다."""
    records = []
    for line_number, raw_line in enumerate(
        Path(label_path).read_text(encoding="utf-8").splitlines(), start=1
    ):
        if not raw_line.strip():
            continue
        fields = raw_line.split()
        if len(fields) != 5:
            raise ValueError(f"Invalid YOLO label format: {label_path}:{line_number}")

        class_value = float(fields[0])
        if not math.isfinite(class_value) or not class_value.is_integer():
            raise ValueError(f"Non-integer YOLO class ID: {label_path}:{line_number}")
        class_id = int(class_value)

        center_x, center_y, width, height = map(float, fields[1:])
        if not 0 <= class_id < EXPECTED_NUM_CLASSES:
            raise ValueError(f"Out-of-range YOLO class ID: {label_path}:{line_number}")
        if not all(math.isfinite(value) for value in [center_x, center_y, width, height]):
            raise ValueError(f"Non-finite YOLO coordinate: {label_path}:{line_number}")
        if not (
            0 <= center_x <= 1 and 0 <= center_y <= 1
            and 0 < width <= 1 and 0 < height <= 1
        ):
            raise ValueError(f"Out-of-range YOLO coordinate: {label_path}:{line_number}")
        if center_x - width / 2 < -1e-5 or center_x + width / 2 > 1 + 1e-5:
            raise ValueError(f"YOLO x extent exceeds image: {label_path}:{line_number}")
        if center_y - height / 2 < -1e-5 or center_y + height / 2 > 1 + 1e-5:
            raise ValueError(f"YOLO y extent exceeds image: {label_path}:{line_number}")
        records.append((class_id, center_x, center_y, width, height))
    return records


def index_bundle_files(root, suffixes):
    """train/val 중첩 구조를 재귀 인덱싱하고 파일명 중복을 차단한다."""
    index = {}
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file() or path.suffix.lower() not in suffixes:
            continue
        if path.name in index:
            raise ValueError(f"Duplicate file name inside the bundle: {path.name}")
        index[path.name] = path
    return index


def _expected_split_from_path(path, root):
    relative = Path(path).relative_to(root)
    if len(relative.parts) < 2:
        raise RuntimeError(f"Expected split subdirectory under {root}: {path}")
    return relative.parts[0].lower()


def load_and_validate_yolo_bundle(track_name, bundle_dir):
    """YOLO11 V6.0.5 FINAL 이미지·라벨·mapping·manifest 계약을 전수 검사한다."""
    bundle_dir = Path(bundle_dir)
    image_dir = bundle_dir / "images"
    label_dir = bundle_dir / "labels"
    mapping_path = bundle_dir / "class_mapping.csv"

    report_path, report = read_preprocessing_report(bundle_dir)
    manifest_df = read_dataset_manifest(bundle_dir)
    mapping_df, idx_to_original = read_yolo_mapping(mapping_path)

    if len(mapping_df) != EXPECTED_NUM_CLASSES:
        raise ValueError(f"Expected {EXPECTED_NUM_CLASSES} classes, found {len(mapping_df)}.")

    canonical_file_names = sorted(manifest_df["file_name"].astype(str))
    image_index = index_bundle_files(
        image_dir, {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
    )
    label_index = index_bundle_files(label_dir, {".txt"})

    # [검토2 수정] 라벨은 stem(확장자 제거)으로 매칭되므로 abc.jpg와 abc.png가 함께 있으면
    # 서로 다른 두 이미지가 같은 abc.txt 한 개를 공유하게 된다. 파일명(확장자 포함) 중복은
    # index_bundle_files가 이미 막지만 stem 충돌은 못 잡으므로 여기서 명시적으로 차단한다.
    _stem_owner = {}
    for _image_name in image_index:
        _stem = Path(_image_name).stem
        if _stem in _stem_owner:
            raise ValueError(
                f"Image stem collision: '{_stem_owner[_stem]}' and '{_image_name}' "
                f"would share label '{_stem}.txt'."
            )
        _stem_owner[_stem] = _image_name

    if set(image_index) != set(canonical_file_names):
        raise ValueError("Bundle image files and dataset_manifest.csv do not match.")

    expected_label_names = {f"{Path(name).stem}.txt" for name in canonical_file_names}
    if set(label_index) != expected_label_names:
        raise ValueError("Bundle label files and dataset_manifest.csv do not match.")

    manifest_by_file = manifest_df.set_index("file_name")
    total_objects = 0
    label_records = {}

    # V6.0.5 manifest에 이미 게시 이미지 SHA-256이 있으므로 원본 이미지를 여기서 또 해싱하지 않는다.
    image_hashes = {
        str(row.file_name): str(row.image_sha256)
        for row in manifest_df.itertuples()
    }

    for file_name in tqdm(
        canonical_file_names,
        desc=f"[1-1] Validating {track_name} V6.0.5",
        unit="image",
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        expected_split = str(manifest_by_file.loc[file_name, "split"]).lower()
        image_path = image_index[file_name]
        label_name = f"{Path(file_name).stem}.txt"
        label_path = label_index[label_name]

        if _expected_split_from_path(image_path, image_dir) != expected_split:
            raise RuntimeError(
                f"Image split mismatch: {file_name} is not under images/{expected_split}/"
            )
        if _expected_split_from_path(label_path, label_dir) != expected_split:
            raise RuntimeError(
                f"Label split mismatch: {label_name} is not under labels/{expected_split}/"
            )

        with Image.open(image_path) as image:
            if image.size != (EXPECTED_TARGET_SIZE, EXPECTED_TARGET_SIZE):
                raise ValueError(
                    f"Unexpected {track_name} image size: {file_name} -> {image.size}"
                )
            image.verify()

        records = validate_yolo_label_file(label_path)
        expected_object_count = int(manifest_by_file.loc[file_name, "object_count"])
        if len(records) != expected_object_count:
            raise RuntimeError(
                f"Object count mismatch for {file_name}: labels={len(records)}, "
                f"manifest={expected_object_count}"
            )
        label_records[file_name] = records
        total_objects += len(records)

    if total_objects != EXPECTED_OBJECTS:
        raise ValueError(
            f"Expected {EXPECTED_OBJECTS} objects, found {total_objects}."
        )

    observed_class_ids = {
        class_id for records in label_records.values() for class_id, *_ in records
    }
    if observed_class_ids != set(range(EXPECTED_NUM_CLASSES)):
        missing = sorted(set(range(EXPECTED_NUM_CLASSES)) - observed_class_ids)
        extra = sorted(observed_class_ids - set(range(EXPECTED_NUM_CLASSES)))
        raise RuntimeError(f"Class coverage mismatch: missing={missing}, extra={extra}")

    print(
        f"{track_name} V6.0.5 bundle verified: "
        f"{len(canonical_file_names):,} images / {total_objects:,} objects / "
        f"{len(mapping_df)} classes."
    )
    # [검토2 수정] report의 group/label_contract_sha256은 전처리 중간 산출물로 계산되어 번들에
    # 그 원본이 포함되지 않으므로 여기서 그대로 재현할 수 없다. 대신 실제로 받은 manifest의
    # per-file image/label sha256을 정렬·집계한 독립 무결성 지문을 계산해 기록한다. 파일별 해시는
    # staging(1-5 셀)에서 재해싱으로 대조되므로, 이 집계 지문이 흔들리면 받은 데이터가 바뀐 것이다.
    _integrity_rows = sorted(
        (str(row.file_name), str(row.image_sha256), str(row.label_sha256))
        for row in manifest_df.itertuples()
    )
    received_manifest_hash_digest = hashlib.sha256(
        "\n".join(f"{fn}\t{ih}\t{lh}" for fn, ih, lh in _integrity_rows).encode("utf-8")
    ).hexdigest()

    return {
        "received_manifest_hash_digest": received_manifest_hash_digest,
        "track_name": track_name,
        "bundle_dir": bundle_dir,
        "image_dir": image_dir,
        "label_dir": label_dir,
        "image_paths": image_index,
        "label_paths": label_index,
        "mapping_path": mapping_path,
        "mapping_df": mapping_df,
        "idx_to_original": idx_to_original,
        "manifest_df": manifest_df,
        "report_path": report_path,
        "report": report,
        "image_hashes": image_hashes,
        "label_records": label_records,
        "canonical_file_names": canonical_file_names,
    }


def build_coco_like_from_yolo(yolo_track_data):
    """검증된 YOLO 라벨을 공통 평가용 COCO-like 구조로 변환한다."""
    mapping_df = yolo_track_data["mapping_df"]
    categories = [
        {
            "id": int(row.yolo_class_id),
            "original_category_id": int(row.original_category_id),
            "name": str(row.normalized_class_name),
        }
        for row in mapping_df.itertuples()
    ]

    file_names = yolo_track_data["canonical_file_names"]
    images = [
        {"id": index, "file_name": file_name}
        for index, file_name in enumerate(file_names)
    ]
    file_name_to_id = {image["file_name"]: image["id"] for image in images}

    annotations = []
    for file_name, records in yolo_track_data["label_records"].items():
        image_id = file_name_to_id[file_name]
        for class_id, center_x, center_y, width, height in records:
            px_width = width * EXPECTED_TARGET_SIZE
            px_height = height * EXPECTED_TARGET_SIZE
            px_x = center_x * EXPECTED_TARGET_SIZE - px_width / 2
            px_y = center_y * EXPECTED_TARGET_SIZE - px_height / 2
            annotations.append({
                "id": len(annotations),
                "image_id": image_id,
                "category_id": class_id,
                "bbox": [px_x, px_y, px_width, px_height],
            })
    return {"images": images, "annotations": annotations, "categories": categories}


def build_file_to_group_map(yolo_track_data):
    manifest_df = yolo_track_data["manifest_df"]
    return dict(zip(
        manifest_df["file_name"].astype(str),
        manifest_df["group_id"].astype(str),
    ))


# ---- 실행 ----
yolo_data = {
    track_name: load_and_validate_yolo_bundle(track_name, bundle_dir)
    for track_name, bundle_dir in YOLO_TRACK_DIRS.items()
}
canonical_data = yolo_data[SHARED_BUNDLE_KEY]
canonical_file_names = canonical_data["canonical_file_names"]

coco_like = build_coco_like_from_yolo(canonical_data)
file_to_group = build_file_to_group_map(canonical_data)
UPSTREAM_SPLIT_BY_GROUP = build_upstream_split_by_group(canonical_data["manifest_df"])

_upstream_counts = Counter(UPSTREAM_SPLIT_BY_GROUP.values())
if set(_upstream_counts) != {"train", "val"}:
    raise RuntimeError(f"Unexpected upstream group split values: {dict(_upstream_counts)}")
print(f"Upstream V6.0.5 group split: {dict(_upstream_counts)}")

split_files, class_groups, split_score, split_groups, upstream_class_coverage_table = (
    build_exact_upstream_split(
        coco_like,
        file_to_group,
        canonical_data["manifest_df"],
    )
)

canonical_dataset_fingerprint = stable_json_hash({
    "image_hashes": canonical_data["image_hashes"],
    "mapping": canonical_data["mapping_df"].to_dict("records"),
    "manifest": canonical_data["manifest_df"].to_dict("records"),
    "pipeline_contract": YOLO11_V605_CONTRACT,
})

canonical_dataset = {
    "coco": coco_like,
    "dataset_fingerprint": canonical_dataset_fingerprint,
    "report": canonical_data["report"],
    "image_dir": canonical_data["image_dir"],
    "image_hashes": canonical_data["image_hashes"],
    "manifest_df": canonical_data["manifest_df"],
}
canonical_group_map = file_to_group

_coco_like_image_name_by_id = {
    image["id"]: image["file_name"] for image in coco_like["images"]
}
_yolo_id_to_original_id = {
    category["id"]: category["original_category_id"]
    for category in coco_like["categories"]
}
coco_original_annotations = defaultdict(list)
for annotation in coco_like["annotations"]:
    _file_name = _coco_like_image_name_by_id[annotation["image_id"]]
    _x, _y, _w, _h = annotation["bbox"]
    _original_id = _yolo_id_to_original_id[annotation["category_id"]]
    coco_original_annotations[_file_name].append(
        (_original_id, [_x, _y, _x + _w, _y + _h])
    )


[1-1] Validating shared_v605_bundle V6.0.5:   0%|          | 0/12196 [00:00<?, ?image/s]

shared_v605_bundle V6.0.5 bundle verified: 12,196 images / 46,394 objects / 118 classes.
Upstream V6.0.5 group split: {'train': 3283, 'val': 821}
=== YOLO11 V6.0.5 upstream split preserved exactly ===
Train: 3,283 groups / 9,763 images
Val  : 821 groups / 2,433 images
Independent Test: not provided; no downstream split was fabricated.


,category_id,group_count,train_objects,val_objects,is_single_group_class,validation_status
0,0,57,149,21,False,observed
1,1,78,181,51,False,observed
2,2,89,109,44,False,observed
3,3,15,33,12,False,observed
4,4,86,224,32,False,observed
...,...,...,...,...,...,...
113,113,123,311,57,False,observed
114,114,114,282,60,False,observed
115,115,124,264,101,False,observed
116,116,127,330,50,False,observed


## 1-4. 고정 Train / Validation 확정과 클래스 커버리지 보고

YOLO11 V6.0.5 FINAL이 제공한 split을 그대로 확정한다.
이 단계에서는 랜덤 split 탐색이나 Test 생성이 없다.


In [7]:
# YOLO11 V6.0.5 upstream split을 그대로 기록한다.
split_manifest = {
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_code_version": PIPELINE_CODE_VERSION,
    "split_policy": SPLIT_POLICY,
    "independent_test_available": False,
    "train_files": split_files["train"],
    "val_files": split_files["val"],
    "train_groups": sorted(split_groups["train"]),
    "val_groups": sorted(split_groups["val"]),
    "source_group_contract_sha256": canonical_dataset["report"]["group_contract_sha256"],
}
split_fingerprint = stable_json_hash(split_manifest)
split_manifest_path = MANIFEST_DIR / f"preserved_split_{split_fingerprint[:16]}.json"
write_json_atomic(split_manifest_path, split_manifest)

class_coverage_table = upstream_class_coverage_table.copy()
class_coverage_path = REPORT_DIR / f"class_split_coverage_{split_fingerprint[:16]}.csv"
class_coverage_table.to_csv(class_coverage_path, index=False, encoding="utf-8-sig")

val_observed_class_count = int(class_coverage_table["val_objects"].gt(0).sum())

print("Fixed YOLO11 V6.0.5 split:")
print(
    f"Train {len(split_files['train']):,} images / "
    f"Validation {len(split_files['val']):,} images."
)
print(
    f"Validation class coverage: {val_observed_class_count}/{EXPECTED_NUM_CLASSES} "
    f"({val_observed_class_count / EXPECTED_NUM_CLASSES:.1%})."
)
print("Independent Test: unavailable; no Validation/Test re-split was performed.")
display(class_coverage_table)


Fixed YOLO11 V6.0.5 split:
Train 9,763 images / Validation 2,433 images.
Validation class coverage: 118/118 (100.0%).
Independent Test: unavailable; no Validation/Test re-split was performed.


,category_id,group_count,train_objects,val_objects,is_single_group_class,validation_status
0,0,57,149,21,False,observed
1,1,78,181,51,False,observed
2,2,89,109,44,False,observed
3,3,15,33,12,False,observed
4,4,86,224,32,False,observed
...,...,...,...,...,...,...
113,113,123,311,57,False,observed
114,114,114,282,60,False,observed
115,115,124,264,101,False,observed
116,116,127,330,50,False,observed


In [8]:
# V6.0.5: 랜덤 split 민감도 진단을 제거한다.
# upstream Train/Val은 이미 고정 계약이므로 seed를 바꿔 다시 나누는 것 자체가 계약 위반이다.

def observed_class_ids_for_files(coco, file_names):
    image_id_by_name = {image["file_name"]: int(image["id"]) for image in coco["images"]}
    selected_ids = {image_id_by_name[name] for name in file_names}
    return {
        int(annotation["category_id"])
        for annotation in coco["annotations"]
        if int(annotation["image_id"]) in selected_ids
    }


preserved_split_audit_table = pd.DataFrame([
    {
        "split": "train",
        "images": len(split_files["train"]),
        "groups": len(split_groups["train"]),
        "observed_classes": len(
            observed_class_ids_for_files(canonical_dataset["coco"], split_files["train"])
        ),
    },
    {
        "split": "val",
        "images": len(split_files["val"]),
        "groups": len(split_groups["val"]),
        "observed_classes": len(
            observed_class_ids_for_files(canonical_dataset["coco"], split_files["val"])
        ),
    },
])
PRESERVED_SPLIT_AUDIT_PATH = (
    MANIFEST_DIR / f"preserved_upstream_split_audit_{split_fingerprint[:16]}.csv"
)
preserved_split_audit_table.to_csv(
    PRESERVED_SPLIT_AUDIT_PATH, index=False, encoding="utf-8-sig"
)
display(preserved_split_audit_table)

# 원본 manifest의 보존 사본. Train/Val을 합친 'full-data final training' manifest는 만들지 않는다.
preserved_manifest = canonical_dataset["manifest_df"].copy().sort_values("file_name").reset_index(drop=True)
PRESERVED_SOURCE_MANIFEST_PATH = (
    MANIFEST_DIR / f"v605_source_manifest_{split_fingerprint[:16]}.csv"
)
preserved_manifest.to_csv(
    PRESERVED_SOURCE_MANIFEST_PATH, index=False, encoding="utf-8-sig"
)
if len(preserved_manifest) != EXPECTED_IMAGES:
    raise RuntimeError(f"Manifest count mismatch: {len(preserved_manifest)}")

# EXIF 관련 열이 전달된 경우 좌표계 변경 여부를 확인한다.
exif_changed_columns = [
    column for column in preserved_manifest.columns
    if column.lower() in {"orientation_changed", "exif_orientation_changed"}
]
if exif_changed_columns:
    exif_changed_count = int(
        preserved_manifest[exif_changed_columns[0]].fillna(False).astype(bool).sum()
    )
else:
    exif_changed_count = 0
if exif_changed_count > 0:
    raise RuntimeError(
        f"EXIF orientation changed images detected: {exif_changed_count}. "
        "Confirm BBox coordinate frame before training."
    )
print(f"EXIF BBox coordinate-frame gate: PASS ({exif_changed_count} changed images)")

IMAGE_CACHE_ABLATION_NOTE = {
    "source": "YOLO11 V6.0.5 preprocessed 960x960 images",
    "staging": "copy/hard-link only; no additional image preprocessing or recompression",
}
print(IMAGE_CACHE_ABLATION_NOTE)


,split,images,groups,observed_classes
0,train,9763,3283,118
1,val,2433,821,118


EXIF BBox coordinate-frame gate: PASS (0 changed images)
{'source': 'YOLO11 V6.0.5 preprocessed 960x960 images', 'staging': 'copy/hard-link only; no additional image preprocessing or recompression'}


## 1-5. 공통 로컬 학습 데이터 준비

V6.0.5 FINAL의 이미지와 YOLO 라벨을 **수정 없이** 로컬로 staging한다.
세 모델은 같은 `images/train`, `images/val`, `labels/train`, `labels/val`을 공유한다.

원본 class mapping도 그대로 사용하며, 로컬 staging 후 이미지/라벨 SHA-256을
`dataset_manifest.csv`와 대조한다.


In [9]:
def build_canonical_yolo_mapping(mapping_df):
    """V6.0.5 class_mapping.csv를 그대로 공통 YOLO mapping으로 사용한다."""
    mapping_df = mapping_df.sort_values("yolo_class_id").reset_index(drop=True)
    original_to_yolo = {
        int(row.original_category_id): int(row.yolo_class_id)
        for row in mapping_df.itertuples()
    }
    names = {
        int(row.yolo_class_id): str(row.normalized_class_name)
        for row in mapping_df.itertuples()
    }
    if sorted(names) != list(range(EXPECTED_NUM_CLASSES)):
        raise RuntimeError("Source class mapping is not continuous.")
    return original_to_yolo, names


def _link_or_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)


def stage_shared_training_dataset():
    """V6.0.5의 고정 Train/Val 이미지·라벨을 내용 변경 없이 로컬에 staging한다."""
    reset_owned_directory(LOCAL_IMAGE_ROOT, LOCAL_WORK_ROOT)
    reset_owned_directory(LOCAL_YOLO_DATASET, LOCAL_WORK_ROOT)

    source_manifest = canonical_data["manifest_df"].set_index("file_name")
    original_to_yolo, names = build_canonical_yolo_mapping(canonical_data["mapping_df"])

    staged_image_hashes = {}
    staged_label_hashes = {}

    for split_name in ["train", "val"]:
        image_split_dir = LOCAL_YOLO_DATASET / "images" / split_name
        label_split_dir = LOCAL_YOLO_DATASET / "labels" / split_name
        image_split_dir.mkdir(parents=True, exist_ok=True)
        label_split_dir.mkdir(parents=True, exist_ok=True)

        for file_name in tqdm(
            split_files[split_name],
            desc=f"[1-5] Staging V6.0.5 {split_name}",
            unit="image",
            bar_format=PROGRESS_BAR_FORMAT,
        ):
            source_image = canonical_data["image_paths"][file_name]
            source_label = canonical_data["label_paths"][f"{Path(file_name).stem}.txt"]

            # shared_images는 시각화/공통 접근용. 실제 학습 폴더는 split별 hard-link/copy.
            shared_image = LOCAL_IMAGE_ROOT / file_name
            _link_or_copy(source_image, shared_image)

            destination_image = image_split_dir / file_name
            destination_label = label_split_dir / f"{Path(file_name).stem}.txt"
            _link_or_copy(shared_image, destination_image)
            _link_or_copy(source_label, destination_label)

            staged_image_hashes[file_name] = sha256_file(destination_image)
            staged_label_hashes[file_name] = sha256_file(destination_label)

            manifest_row = source_manifest.loc[file_name]
            if staged_image_hashes[file_name] != str(manifest_row["image_sha256"]):
                raise RuntimeError(f"Staged image hash mismatch: {file_name}")
            if staged_label_hashes[file_name] != str(manifest_row["label_sha256"]):
                raise RuntimeError(f"Staged label hash mismatch: {file_name}")

    # [검토2 수정] 원본 data.yaml을 필수로 요구만 하고 읽지 않으면, 전처리 번들의 nc/names가
    # class_mapping.csv와 어긋나 있어도(둘 다 전처리 산출물이지만 갱신 시점이 다를 수 있다) 그대로
    # 넘어간다. 로컬 data.yaml을 새로 쓰기 전에 원본 data.yaml을 읽어 계약(nc/names/train/val)과 대조한다.
    _source_data_yaml_path = REQUIRED_SOURCE_FILES["data.yaml"]
    with _source_data_yaml_path.open("r", encoding="utf-8") as _syf:
        _source_data_yaml = yaml.safe_load(_syf) or {}
    if int(_source_data_yaml.get("nc", -1)) != EXPECTED_NUM_CLASSES:
        raise RuntimeError(
            f"Source data.yaml nc={_source_data_yaml.get('nc')!r} != contract {EXPECTED_NUM_CLASSES}."
        )
    _source_names = _source_data_yaml.get("names", {})
    if isinstance(_source_names, list):
        _source_names = {index: value for index, value in enumerate(_source_names)}
    _source_names = {int(key): str(value) for key, value in _source_names.items()}
    if _source_names and _source_names != {int(k): str(v) for k, v in names.items()}:
        raise RuntimeError(
            "Source data.yaml names do not match the verified class_mapping.csv names."
        )
    for _split_key in ("train", "val"):
        if _split_key not in _source_data_yaml:
            raise RuntimeError(f"Source data.yaml is missing the '{_split_key}' entry.")
    if "test" in _source_data_yaml:
        raise RuntimeError("Source data.yaml unexpectedly declares a 'test' split (contract is train/val only).")

    # source data.yaml의 class 의미는 유지하되 local path만 고정한다.
    data_yaml = {
        "path": str(LOCAL_YOLO_DATASET),
        "train": "images/train",
        "val": "images/val",
        "nc": EXPECTED_NUM_CLASSES,
        "names": names,
        "dataset_state": "presplit_train_val_labeled_dataset",
        "split_policy": "preserve_exact_upstream",
        "independent_test_available": False,
    }
    data_yaml_path = LOCAL_YOLO_DATASET / "data.yaml"
    with data_yaml_path.open("w", encoding="utf-8") as file:
        yaml.safe_dump(data_yaml, file, allow_unicode=True, sort_keys=False)

    # class_mapping.csv도 원본 그대로 로컬에 보존한다.
    shutil.copy2(
        canonical_data["mapping_path"],
        LOCAL_YOLO_DATASET / "class_mapping.csv",
    )
    canonical_data["manifest_df"].to_csv(
        LOCAL_YOLO_DATASET / "dataset_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return data_yaml_path, original_to_yolo, names


data_yaml_path, ORIGINAL_TO_YOLO, YOLO_NAMES = stage_shared_training_dataset()
YOLO_TO_ORIGINAL = {
    yolo_id: original_id for original_id, yolo_id in ORIGINAL_TO_YOLO.items()
}

yolo_class_reference_table = pd.DataFrame([
    {
        "yolo_class_id": yolo_id,
        "original_category_id": YOLO_TO_ORIGINAL[yolo_id],
        "class_name": YOLO_NAMES[yolo_id],
    }
    for yolo_id in sorted(YOLO_NAMES)
])
yolo_class_reference_path = (
    MANIFEST_DIR / f"yolo_class_reference_{split_fingerprint[:16]}.csv"
)
yolo_class_reference_table.to_csv(
    yolo_class_reference_path, index=False, encoding="utf-8-sig"
)

print(f"Shared local image directory: {LOCAL_IMAGE_ROOT}")
print(f"Shared YOLO V6.0.5 dataset: {data_yaml_path}")
print(f"YOLO class reference: {yolo_class_reference_path}")


[1-5] Staging V6.0.5 train:   0%|          | 0/9763 [00:00<?, ?image/s]

[1-5] Staging V6.0.5 val:   0%|          | 0/2433 [00:00<?, ?image/s]

Shared local image directory: /content/baby_kangaroo_week2_common_pipeline_v605/shared_images
Shared YOLO V6.0.5 dataset: /content/baby_kangaroo_week2_common_pipeline_v605/yolo_common/data.yaml
YOLO class reference: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/manifests/yolo_class_reference_cc5d16d3fe042c52.csv


# 2. 모델함수

## 2-0. 초기 학습과 파인튜닝 계획

기존 파이프라인은 모델별 학습 함수를 억지로 한 함수에 합치지 않고, 각 모델의 학습 함수가 비슷한 입출력 형태를 갖도록 정리했다. 이 구조 덕분에 세 모델의 실행 결과를 같은 평가 단계로 넘기는 흐름을 편하게 이어갈 수 있었다.

이 구조를 유지하면서 초기 학습 뒤에 모델별 파인튜닝 후보를 추가한다. 초기 학습은 세 모델의 출발점을 확인하는 실험이고, 그다음 모델마다 학습률, batch, epoch가 다른 두 후보를 학습한다. 파인튜닝 후보에는 Train 전용 안전 증강을 추가하고, Validation mAP50-95가 좋아질 때만 best checkpoint를 갱신한다.

`standard`에서는 아래 표의 설정을 그대로 실행한다. `smoke`에서는 각 실험을 1 epoch만 실행해 연결 오류를 빠르게 찾는다.


각 실험은 시작과 종료 시각, 실제 경과시간을 자동으로 기록한다. 실험별 JSON은 학습 직후 `timings/`에 저장하고, 마지막에는 모델별 누적시간을 CSV로 비교한다. 기존 checkpoint를 재사용한 경우에는 재사용에 걸린 시간과 원래 학습시간을 구분한다.


In [10]:
# 초기 학습은 과도한 증강 없이 같은 출발 조건을 확인한다.
# [V5 정리] 세 모델 모두 epochs와 early_stopping_patience가 12로 동일하다.
# patience가 총 epoch 수와 같으면(patience >= epochs) EarlyStopping은 학습이 끝나기
# 전에 발동할 수 없으므로, 이는 "baseline 단계에서는 세 모델을 정확히 같은 epoch 수만큼만
# 학습시켜 출발 조건을 동일하게 맞춘다"는 의도의 조기 종료 비활성화로 읽어야 한다.
# (아래 TUNING_CONFIGS는 patience=10 < epochs로 실제 조기 종료가 동작한다.)
# 값 자체는 바꾸지 않는다 - patience를 epochs보다 작게 줄이면 baseline 비교의 전제인
# "세 모델 동일 epoch" 조건이 깨지기 때문이다.
BASELINE_CONFIGS = {
    "YOLO11s": {
        "experiment_id": "yolo11s_baseline",
        "stage": "baseline",
        "epochs": 12,
        "batch_size": 2,
        "learning_rate": 0.005,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "augmentation": "baseline",
        "early_stopping_patience": 12,
    },
    "YOLO11m": {
        "experiment_id": "yolo11m_baseline",
        "stage": "baseline",
        "epochs": 12,
        "batch_size": 2,
        "learning_rate": 0.005,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "augmentation": "baseline",
        "early_stopping_patience": 12,
    },
    "YOLO12m": {
        "experiment_id": "yolo12m_baseline",
        "stage": "baseline",
        "epochs": 12,
        "batch_size": 2,
        "learning_rate": 0.005,
        "momentum": 0.9,
        "weight_decay": 0.0005,
        "augmentation": "baseline",
        "early_stopping_patience": 12,
    },
}

# 모델별로 두 파인튜닝 후보를 두어 학습률·batch·epoch 영향을 실제로 비교한다.
# YOLO11s에서 확인한 최적 구간(epoch 40 부근)을 YOLO11m·YOLO12m tune_a의 출발점으로 삼는다.
TUNING_CONFIGS = {
    "YOLO11s": [
        {
            "experiment_id": "yolo11s_tune_a",
            "stage": "tuning",
            "epochs": 35,
            "batch_size": 4,
            "learning_rate": 0.0025,
            "momentum": 0.9,
            "weight_decay": 0.0005,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
        {
            "experiment_id": "yolo11s_tune_b",
            "stage": "tuning",
            "epochs": 42,
            "batch_size": 2,
            "learning_rate": 0.005,
            "momentum": 0.9,
            "weight_decay": 0.0001,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
    ],
    "YOLO11m": [
        {
            "experiment_id": "yolo11m_tune_a",
            "stage": "tuning",
            "epochs": 40,
            "batch_size": 2,
            "learning_rate": 0.0025,
            "momentum": 0.9,
            "weight_decay": 0.0005,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
        {
            "experiment_id": "yolo11m_tune_b",
            "stage": "tuning",
            "epochs": 48,
            "batch_size": 2,
            "learning_rate": 0.0040,
            "momentum": 0.9,
            "weight_decay": 0.0001,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
    ],
    "YOLO12m": [
        {
            "experiment_id": "yolo12m_tune_a",
            "stage": "tuning",
            "epochs": 40,
            "batch_size": 2,
            "learning_rate": 0.0025,
            "momentum": 0.9,
            "weight_decay": 0.0005,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
        {
            "experiment_id": "yolo12m_tune_b",
            "stage": "tuning",
            "epochs": 48,
            "batch_size": 2,
            "learning_rate": 0.0040,
            "momentum": 0.9,
            "weight_decay": 0.0001,
            "augmentation": "safe_detection",
            "early_stopping_patience": 10,
        },
    ],
}


# [V6 추가] 위 BASELINE/TUNING 설정은 12,196장짜리 데이터에 맞춰 잡은 값이다(batch=2, epochs 12~48).
# 12,000장짜리 번들에서 batch=2는 epoch당 6,000 step이 되고, epochs 48이면 단일 실험만
# 30만 step이 넘는다. Colab 세션 안에서 9개 실험을 끝낼 수 없는 규모다.
# 데이터가 커지면 batch를 키우고 epoch를 줄인 값으로 자동 보정하고, 무엇을 바꿨는지 표로 남긴다.
# (자동 보정을 원치 않으면 SCALE_TRAINING_TO_DATASET = False로 끄면 된다.)
SCALE_TRAINING_TO_DATASET = True
LARGE_DATASET_IMAGE_THRESHOLD = 2000
# [검토2 수정] 이전 override는 모든 tuning 후보를 epochs=16/batch=8로 '절대값 덮어쓰기'해서
# tune A(epochs 35)/tune B(epochs 42)의 epoch 비교축과 batch 비교축을 통째로 없앴다. 그러면
# 실제 비교 변수는 lr/weight_decay만 남는데 문서·quality check는 여전히 epoch/batch ablation처럼
# 말한다. 대신 후보별 원래 값에 공통 '스케일'을 곱해 상대 차이를 보존하고, epoch 상한만 둔다.
# batch는 모델 크기별 상한을 둬서 m급+960 OOM을 사전 방어한다(s는 조금 크게 허용).
LARGE_DATASET_EPOCH_SCALE = 0.4          # 12k장 규모에서 epoch 수를 40%로 축소(상대 차이는 유지)
LARGE_DATASET_MAX_EPOCHS = 20            # 축소 후에도 이 값을 넘지 않게 상한
LARGE_DATASET_MIN_EPOCHS = 6
LARGE_DATASET_BATCH_CAP_BY_SCALE = {"s": 8, "m": 4}  # 960px에서 m급은 batch 상한을 낮춘다
LARGE_DATASET_TUNING_PATIENCE = 5
LARGE_DATASET_BASELINE_PATIENCE = None   # None이면 baseline은 원래대로 patience=epochs(조기종료 비활성)


def _model_scale_hint(config):
    """experiment_id 접두사(yolo11s/yolo11m/yolo12m)로 배치 상한용 모델 크기를 추정한다."""
    experiment_id = str(config.get("experiment_id", ""))
    return "m" if ("11m" in experiment_id or "12m" in experiment_id) else "s"


def effective_config(config):
    """대형 데이터에서 후보 간 상대 차이를 보존한 채 epoch/batch를 스케일하고,
    smoke 모드에서만 epoch와 patience를 1로 줄인다. 원본 설정 자체는 보존한다."""
    resolved = dict(config)
    if (
        SCALE_TRAINING_TO_DATASET
        and EXPECTED_IMAGES is not None
        and EXPECTED_IMAGES >= LARGE_DATASET_IMAGE_THRESHOLD
    ):
        original_epochs = int(resolved["epochs"])
        scaled_epochs = int(round(original_epochs * LARGE_DATASET_EPOCH_SCALE))
        scaled_epochs = max(LARGE_DATASET_MIN_EPOCHS, min(LARGE_DATASET_MAX_EPOCHS, scaled_epochs))
        resolved["epochs"] = scaled_epochs

        batch_cap = LARGE_DATASET_BATCH_CAP_BY_SCALE[_model_scale_hint(resolved)]
        resolved["batch_size"] = min(int(resolved["batch_size"]), batch_cap)

        if resolved["stage"] == "tuning":
            resolved["early_stopping_patience"] = min(
                int(resolved["early_stopping_patience"]), LARGE_DATASET_TUNING_PATIENCE
            )
        elif LARGE_DATASET_BASELINE_PATIENCE is None:
            # baseline은 '세 모델 동일 epoch' 전제를 위해 patience=epochs(조기종료 비활성)를 유지한다.
            resolved["early_stopping_patience"] = scaled_epochs
        else:
            resolved["early_stopping_patience"] = LARGE_DATASET_BASELINE_PATIENCE
        resolved["scaled_for_large_dataset"] = True
    else:
        resolved["scaled_for_large_dataset"] = False
    if RUN_PROFILE == "smoke":
        resolved["epochs"] = 1
        resolved["early_stopping_patience"] = 1
    resolved["image_size"] = IMAGE_SIZE
    resolved["run_profile"] = RUN_PROFILE
    return resolved



TIMING_SCHEMA_VERSION = 1


def format_duration(seconds):
    """초 단위 시간을 사람이 바로 읽을 수 있는 시:분:초 문자열로 바꾼다."""
    seconds = max(0.0, float(seconds))
    hours, remainder = divmod(int(round(seconds)), 3600)
    minutes, seconds_part = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds_part:02d}"


def run_timed_experiment(timer_model_name, timer_raw_config, train_callable, *args, **kwargs):
    """학습 함수의 시작·종료·실패 시간을 기록하고 실험별 JSON을 즉시 저장한다."""
    resolved_config = effective_config(timer_raw_config)
    experiment_id = resolved_config["experiment_id"]
    started_at = datetime.now()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started_perf = time.perf_counter()
    print(f"Timer started: {timer_model_name} / {experiment_id} / {started_at.isoformat()}")

    try:
        result = train_callable(*args, **kwargs)
    except Exception as error:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_seconds = time.perf_counter() - started_perf
        finished_at = datetime.now()
        failed_record = {
            "schema_version": TIMING_SCHEMA_VERSION,
            "model": timer_model_name,
            "experiment_id": experiment_id,
            "stage": resolved_config["stage"],
            "run_profile": RUN_PROFILE,
            "last_execution_status": "failed",
            "last_execution_started_at": started_at.isoformat(),
            "last_execution_finished_at": finished_at.isoformat(),
            "last_execution_elapsed_seconds": elapsed_seconds,
            "last_execution_elapsed_hms": format_duration(elapsed_seconds),
            "error_type": type(error).__name__,
            "error_message": str(error),
        }
        failed_path = TIMING_DIR / (
            f"{experiment_id}_failed_{finished_at.strftime('%Y%m%d_%H%M%S')}.json"
        )
        write_json_atomic(failed_path, failed_record)
        print(f"Timer saved after failure: {failed_path}")
        raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started_perf
    finished_at = datetime.now()
    signature = result["manifest"]["run_signature"]
    timing_path = TIMING_DIR / f"{experiment_id}_{signature}.json"
    checkpoint_reused = bool(result.get("checkpoint_reused", False))

    if checkpoint_reused and timing_path.is_file():
        try:
            timing_record = json.loads(timing_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            timing_record = {}
    else:
        timing_record = {}

    # 실제 학습이 수행된 실행만 원본 학습시간으로 고정한다. 재사용 시간은 별도 필드에 남긴다.
    if not checkpoint_reused:
        timing_record.update({
            "schema_version": TIMING_SCHEMA_VERSION,
            "model": timer_model_name,
            "experiment_id": experiment_id,
            "stage": resolved_config["stage"],
            "run_profile": RUN_PROFILE,
            "run_signature": signature,
            "training_started_at": started_at.isoformat(),
            "training_finished_at": finished_at.isoformat(),
            "training_elapsed_seconds": elapsed_seconds,
            "training_elapsed_minutes": elapsed_seconds / 60.0,
            "training_elapsed_hours": elapsed_seconds / 3600.0,
            "training_elapsed_hms": format_duration(elapsed_seconds),
            "reuse_count": 0,
        })
    else:
        timing_record.setdefault("schema_version", TIMING_SCHEMA_VERSION)
        timing_record.setdefault("model", timer_model_name)
        timing_record.setdefault("experiment_id", experiment_id)
        timing_record.setdefault("stage", resolved_config["stage"])
        timing_record.setdefault("run_profile", RUN_PROFILE)
        timing_record.setdefault("run_signature", signature)
        timing_record.setdefault("training_elapsed_seconds", None)
        timing_record.setdefault("training_elapsed_minutes", None)
        timing_record.setdefault("training_elapsed_hours", None)
        timing_record.setdefault("training_elapsed_hms", None)
        timing_record["reuse_count"] = int(timing_record.get("reuse_count", 0)) + 1

    timing_record.update({
        "checkpoint_reused_on_last_execution": checkpoint_reused,
        "last_execution_status": "checkpoint_reused" if checkpoint_reused else "completed",
        "last_execution_started_at": started_at.isoformat(),
        "last_execution_finished_at": finished_at.isoformat(),
        "last_execution_elapsed_seconds": elapsed_seconds,
        "last_execution_elapsed_hms": format_duration(elapsed_seconds),
        "timing_path": str(timing_path),
    })
    write_json_atomic(timing_path, timing_record)
    result["timing"] = timing_record
    print(
        f"Timer completed: {timer_model_name} / {experiment_id} / "
        f"{format_duration(elapsed_seconds)} / "
        f"status={timing_record['last_execution_status']}"
    )
    print(f"Timer record: {timing_path}")
    return result


# 실행 전에 실험 계획을 표로 확인한다.
experiment_plan_rows = []
for model_name in ["YOLO11s", "YOLO11m", "YOLO12m"]:
    for config in [BASELINE_CONFIGS[model_name], *TUNING_CONFIGS[model_name]]:
        experiment_plan_rows.append({"model": model_name, **effective_config(config)})
experiment_plan_table = pd.DataFrame(experiment_plan_rows)
display(experiment_plan_table[
    ["model", "experiment_id", "stage", "epochs", "batch_size", "learning_rate",
     "weight_decay", "augmentation", "scaled_for_large_dataset"]
])

# 실험 하나가 몇 step인지 미리 보여준다. 세션 시간 안에 끝나는 계획인지 판단하는 근거다.
_train_image_count = len(split_files["train"])
_planned_steps = experiment_plan_table.assign(
    steps_per_epoch=lambda frame: np.ceil(_train_image_count / frame["batch_size"]).astype(int),
)
_planned_steps["total_steps"] = _planned_steps["steps_per_epoch"] * _planned_steps["epochs"]
print(f"Train images: {_train_image_count:,}")
print(f"Total planned optimizer steps across all experiments: {int(_planned_steps['total_steps'].sum()):,}")
display(_planned_steps[["model", "experiment_id", "steps_per_epoch", "epochs", "total_steps"]])


,model,experiment_id,stage,epochs,batch_size,learning_rate,weight_decay,augmentation,scaled_for_large_dataset
0,YOLO11s,yolo11s_baseline,baseline,6,2,0.0050,0.0005,baseline,True
1,YOLO11s,yolo11s_tune_a,tuning,14,4,0.0025,0.0005,safe_detection,True
2,YOLO11s,yolo11s_tune_b,tuning,17,2,0.0050,0.0001,safe_detection,True
3,YOLO11m,yolo11m_baseline,baseline,6,2,0.0050,0.0005,baseline,True
4,YOLO11m,yolo11m_tune_a,tuning,16,2,0.0025,0.0005,safe_detection,True
5,YOLO11m,yolo11m_tune_b,tuning,19,2,0.0040,0.0001,safe_detection,True
6,YOLO12m,yolo12m_baseline,baseline,6,2,0.0050,0.0005,baseline,True
7,YOLO12m,yolo12m_tune_a,tuning,16,2,0.0025,0.0005,safe_detection,True
8,YOLO12m,yolo12m_tune_b,tuning,19,2,0.0040,0.0001,safe_detection,True


Train images: 9,763
Total planned optimizer steps across all experiments: 546,784


,model,experiment_id,steps_per_epoch,epochs,total_steps
0,YOLO11s,yolo11s_baseline,4882,6,29292
1,YOLO11s,yolo11s_tune_a,2441,14,34174
2,YOLO11s,yolo11s_tune_b,4882,17,82994
3,YOLO11m,yolo11m_baseline,4882,6,29292
4,YOLO11m,yolo11m_tune_a,4882,16,78112
5,YOLO11m,yolo11m_tune_b,4882,19,92758
6,YOLO12m,yolo12m_baseline,4882,6,29292
7,YOLO12m,yolo12m_tune_a,4882,16,78112
8,YOLO12m,yolo12m_tune_b,4882,19,92758


## 2-2. YOLO 공통 학습 함수

In [11]:
def analyze_yolo_convergence(results_csv_path):
    """results.csv에서 best epoch, 마지막 추세와 수렴 부족 가능성을 계산한다."""
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [str(column).strip() for column in results_df.columns]
    metric_candidates = ["metrics/mAP50-95(B)", "metrics/mAP50-95"]
    metric_column = next((column for column in metric_candidates if column in results_df.columns), None)
    if metric_column is None:
        raise ValueError("Validation mAP50-95 was not found in the YOLO results.csv file.")

    metric_values = pd.to_numeric(results_df[metric_column], errors="raise").astype(float)
    if metric_values.empty or not np.isfinite(metric_values).all() or (metric_values < 0).any():
        raise RuntimeError("YOLO validation mAP contains invalid values.")
    best_index = int(metric_values.idxmax())
    tail_size = min(5, len(metric_values))
    tail_values = metric_values.iloc[-tail_size:].to_numpy()
    tail_slope = float(np.polyfit(np.arange(tail_size), tail_values, 1)[0]) if tail_size >= 2 else 0.0

    loss_columns = [
        column for column in results_df.columns
        if column.startswith("train/") and column.endswith("_loss")
    ]
    train_loss = (
        results_df[loss_columns].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        if loss_columns else pd.Series(dtype=float)
    )
    near_end = best_index >= max(0, math.floor(len(metric_values) * 0.90) - 1)
    needs_more_epochs = bool(near_end and tail_slope > 0.0005)
    return {
        "best_epoch": best_index + 1,
        "best_val_map50_95": float(metric_values.iloc[best_index]),
        "last_val_map50_95": float(metric_values.iloc[-1]),
        "last_five_slope": tail_slope,
        "needs_more_epochs": needs_more_epochs,
        "train_loss_history": train_loss.astype(float).tolist(),
        "val_map50_95_history": metric_values.tolist(),
    }


def validate_yolo_model_scale(model, expected_scale, model_name):
    """체크포인트 내부 YAML의 모델 scale이 기대한 small인지 확인한다."""
    model_yaml = getattr(getattr(model, "model", None), "yaml", {}) or {}
    detected_scale = model_yaml.get("scale")
    if detected_scale is not None and str(detected_scale) != expected_scale:
        raise ValueError(
            f"{model_name} scale mismatch: expected {expected_scale}, found {detected_scale}."
        )
    print(f"{model_name} scale check: {detected_scale or 'metadata unavailable'}")


def yolo_augmentation_arguments(mode):
    """YOLO Train 전용 증강 설정을 반환한다.
    baseline: 세 모델의 출발 조건을 맞추기 위해 fliplr 외 증강을 사실상 비활성화한다.
    safe_detection: 알약 표면 각인(글자·숫자) 판독성을 해치지 않는 범위에서만
    기하·색상 증강을 소폭 적용한다."""
    # [검토2 수정] 이 데이터셋은 알약 표면 각인(글자·숫자)을 판독하는 과제라 좌우 반전은
    # 'AB' -> 거울상 'BA' 같은 현실에 존재하지 않는 각인을 만들어낸다. 따라서 각인 판독을
    # 방해하지 않는다는 설계 취지에 맞게 baseline과 safe_detection 모두 fliplr=0.0으로 둔다.
    # (baseline은 이제 문서 설명대로 실제로 '증강 없음'이 된다.)
    if mode == "baseline":
        return {
            "hsv_h": 0.0,
            "hsv_s": 0.0,
            "hsv_v": 0.0,
            "degrees": 0.0,
            "translate": 0.0,
            "scale": 0.0,
            "shear": 0.0,
            "perspective": 0.0,
            "flipud": 0.0,
            "fliplr": 0.0,
            "mosaic": 0.0,
            "mixup": 0.0,
            "copy_paste": 0.0,
            "erasing": 0.0,
        }
    if mode == "safe_detection":
        return {
            "hsv_h": 0.008,
            "hsv_s": 0.10,
            "hsv_v": 0.08,
            "degrees": 8.0,
            "translate": 0.04,
            "scale": 0.05,
            "shear": 2.0,
            "perspective": 0.0,
            "flipud": 0.0,
            "fliplr": 0.0,
            "mosaic": 0.0,
            "mixup": 0.0,
            "copy_paste": 0.0,
            "erasing": 0.0,
        }
    raise ValueError(f"Unknown YOLO augmentation mode: {mode}")


def train_yolo_experiment(model_name, model_weights, expected_scale, raw_config):
    """YOLO11s·YOLO11m·YOLO12m 중 한 실험을 학습하고 best checkpoint와 설정을 저장한다."""
    config = effective_config(raw_config)
    experiment_id = config["experiment_id"]
    manifest = build_checkpoint_manifest(
        model_name,
        model_weights,
        canonical_dataset["dataset_fingerprint"],
        split_fingerprint,
        config,
    )
    signature = manifest["run_signature"]
    checkpoint_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_best.pt"
    manifest_path = MANIFEST_DIR / f"{experiment_id}_{signature}.json"
    results_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_results.csv"
    args_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_args.yaml"

    if experiment_id not in FORCE_RETRAIN_EXPERIMENTS and checkpoint_is_compatible(
        checkpoint_path,
        manifest_path,
        manifest,
    ):
        trained_model = YOLO(str(checkpoint_path))
        validate_yolo_model_scale(trained_model, expected_scale, model_name)
        convergence = analyze_yolo_convergence(results_path) if results_path.is_file() else None
        print(f"Reused verified {model_name} checkpoint: {checkpoint_path.name}")
        # [V5 정리] analyze_yolo_convergence()가 계산하는 needs_more_epochs는 지금까지
        # 어디에도 출력되지 않고 convergence 딕셔너리 안에 조용히 버려지고 있었다.
        # tail(마지막 5 epoch) 기울기가 여전히 양수인데 best epoch가 학습 막바지에
        # 나왔다는 뜻이므로, 이 실험이 epoch 부족으로 최종 mAP를 놓쳤을 가능성을
        # 알려준다. 재사용된 체크포인트도 동일하게 경고한다.
        if convergence is not None and convergence.get("needs_more_epochs"):
            print(
                f"  [WARNING] {experiment_id}: validation mAP50-95 was still improving "
                f"near the last epoch (slope={convergence['last_five_slope']:.5f}). "
                "Consider increasing epochs for this candidate."
            )
        return {
            "model": trained_model,
            "model_name": model_name,
            "checkpoint_path": checkpoint_path,
            "manifest_path": manifest_path,
            "manifest": manifest,
            "config": config,
            "checkpoint_reused": True,
            "convergence": convergence,
            "results_path": results_path,
        }

    if checkpoint_path.exists():
        backup_file(checkpoint_path, "incomplete_or_incompatible")

    model = YOLO(model_weights)
    validate_yolo_model_scale(model, expected_scale, model_name)
    run_name = f"{experiment_id}_{signature}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    augmentation_arguments = yolo_augmentation_arguments(config["augmentation"])

    model.train(
        data=str(data_yaml_path),
        epochs=config["epochs"],
        batch=config["batch_size"],
        imgsz=IMAGE_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        optimizer="SGD",
        lr0=config["learning_rate"],
        lrf=0.10,
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
        warmup_epochs=2.0 if config["stage"] == "tuning" else 1.0,
        cos_lr=True,
        patience=config["early_stopping_patience"],
        max_det=MAX_DETECTIONS,
        seed=SEED,
        deterministic=True,
        workers=2,
        cache=False,
        amp=USE_AMP,
        project=str(LOCAL_RUNS_ROOT),
        name=run_name,
        exist_ok=False,
        plots=True,
        save=True,
        verbose=True,
        **augmentation_arguments,
    )

    run_dir = Path(model.trainer.save_dir)
    best_source = run_dir / "weights" / "best.pt"
    results_source = run_dir / "results.csv"
    args_source = run_dir / "args.yaml"
    for required_path in [best_source, results_source, args_source]:
        if not required_path.is_file():
            raise FileNotFoundError(f"Required {model_name} training output was not found: {required_path}")

    shutil.copy2(best_source, checkpoint_path)
    shutil.copy2(results_source, results_path)
    shutil.copy2(args_source, args_path)
    convergence = analyze_yolo_convergence(results_path)
    save_checkpoint_manifest(checkpoint_path, manifest_path, manifest)

    trained_model = YOLO(str(checkpoint_path))
    validate_yolo_model_scale(trained_model, expected_scale, model_name)
    print(
        f"Completed {experiment_id}: best epoch {convergence['best_epoch']}, "
        f"validation mAP50-95 {convergence['best_val_map50_95']:.4f}."
    )
    # [V5 정리] 새로 학습한 경우에도 동일하게 needs_more_epochs를 확인해 출력한다.
    if convergence.get("needs_more_epochs"):
        print(
            f"  [WARNING] {experiment_id}: validation mAP50-95 was still improving "
            f"near the last epoch (slope={convergence['last_five_slope']:.5f}). "
            "Consider increasing epochs for this candidate."
        )
    return {
        "model": trained_model,
        "model_name": model_name,
        "checkpoint_path": checkpoint_path,
        "manifest_path": manifest_path,
        "manifest": manifest,
        "config": config,
        "checkpoint_reused": False,
        "convergence": convergence,
        "results_path": results_path,
    }

# 전용 학습 실행 — YOLO11m tune_b


In [12]:
# 이 노트북은 YOLO11m tune_b 한 개만 학습한다.
yolo11m_experiments = {}
tuning_config = TUNING_CONFIGS["YOLO11m"][1]
assert tuning_config["experiment_id"] == "yolo11m_tune_b"
experiment_id = tuning_config["experiment_id"]
yolo11m_experiments[experiment_id] = run_timed_experiment(
    "YOLO11m",
    tuning_config,
    train_yolo_experiment,
    model_name="YOLO11m",
    model_weights="yolo11m.pt",
    expected_scale="m",
    raw_config=tuning_config,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Dedicated experiment completed: {experiment_id}")


Timer started: YOLO11m / yolo11m_tune_b / 2026-08-18T04:35:03.706965
YOLO11m scale check: m
New https://pypi.org/project/ultralytics/8.4.121 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/baby_kangaroo_week2_common_pipeline_v605/yolo_common/data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=19, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.008, hsv_s=0.1, hsv_v=